# PIV-Q27-P0 v1.3 REVIEW-HARDENED — Qwen3.8-27B Stateful Python REPL + Fresh Exact Train Verifier
## ARC-AGI-2 | 4× NVIDIA L4 | Offline Kaggle | Qwen3.8-27B-FP8 TP=4

This notebook is the **first agentic-coding causal experiment after Q1 Direct**. It intentionally changes only the solver mechanism while preserving the proven Q1 infrastructure: same model/checkpoint, vLLM 0.19.0 TP=4 engine, frozen Proxy-48, EDA features, dependency parity, runtime reserve, checkpoint fingerprinting, provenance and scoring denominator.

### Source basis

Primary source snapshot: `gutfeeling/arc-agi-2-submission` at commit
`b3d4494892c6d85fa2a35d548271fb02ea3e191b`.

The P0 mechanism is source-faithful at the **semantic/invariant level**:

1. one reasoning-model lineage per ARC task;
2. one persistent Python REPL namespace across tool turns;
3. the puzzle is preloaded into the REPL and protected from overwrite/whole-puzzle printing;
4. Python stdout/errors/timeouts become model feedback;
5. the model must finish with one self-contained `solution(grid)` program;
6. the final program is executed again in a **fresh process** and must match **every training pair exactly** before any test output is admitted;
7. test grids are produced by execution of the verified program, never transcribed from prose.

### Explicit LOCAL adaptations for Kaggle 4×L4

- Direct `vllm.LLM.generate(...)` + Qwen3.8 native tool-capable chat template instead of an OpenAI/Responses server.
- Lightweight persistent local Python subprocess instead of Docker/IPyBox.
- Hard process timeout + restricted dangerous filesystem/process APIs.
- Bounded P0 budget: one lineage, 6 Python calls, at most 7 model turns, 2,048 tokens/turn and 12,288 generated tokens/task.
- Tool output sent back to Qwen is compact; full bounded raw tool events are checkpointed.
- No reviewer, no second lineage, no deterministic PTQ repair loop, no NVARC seeding, no judge and no evolution. Those are later experiments.

Default is `PREFLIGHT`. Run `PREFLIGHT -> SMOKE_GPU -> PROXY48` in fresh Kaggle sessions.


### v1.1 source/donor re-audit hardening

After re-comparing P0 against the frozen Q1 donor and the audited PiVOTools/Qwen sources, v1.1 fixes four issues without adding a new solver mechanism:
1. use the full source agentic prompt family rather than a shortened hybrid prompt;
2. reject length-truncated generations before parsing them as a tool/final answer;
3. use per-request generation caps inside a batch so one near-context-limit task cannot starve the whole wave;
4. harden the lightweight local worker against host/Kaggle filesystem and process/network access.

It also validates the actual mounted Qwen multi-turn tool-history rendering during PREFLIGHT.


### v1.2 additional verifier/accounting correction

A second source-level pass found that PiVOTools verifies training examples first and executes test inputs in a separate step. v1.1's single verifier exception boundary could erase a genuinely train-perfect result if test execution later failed. v1.2 separates those states and counts train-perfect rate over the entire frozen active task set, including missing/failed tasks as non-passes.


### v1.3 external-review verification

An independent review of the earlier v1.0 was checked against this v1.2 notebook and the audited PiVOTools source.
Some findings had already been fixed in v1.2 (per-request SamplingParams, multi-tool handling, finish-reason gate,
full source prompt, Qwen history probe, train-vs-test verifier state). The remaining real issues are fixed here:
kernel-level `copy` import, source-style bounded retries, a less starvation-prone per-turn ceiling, source-tolerant grid
normalization, per-test partial output retention, budget-aware finalization, think-aware action parsing, dead-worker restart,
puzzle reset hardening, narrower dunder blocking, `py` fences, and a scripted CPU preflight agent-loop test.

Two review suggestions are intentionally NOT adopted:
- no raw-text continuation after truncation (the PiVOTools source retries the agentic call; it does not splice partial text);
- no switch from XHIGH to MEDIUM (PIV-Q27-P0 is the planned XHIGH causal experiment).


In [ ]:
# =====================================================================================
# CELL 1 — PIV-Q27-P0 CONFIG + OFFLINE ENVIRONMENT + FROZEN SOURCE CONTRACT
# Must run before importing torch/vLLM.
# =====================================================================================
import os
import sys
import json
import math
import time
import random
import hashlib
import zipfile
import shutil
import traceback
import copy
import importlib.metadata
from pathlib import Path
from datetime import datetime, timezone
from collections import deque

SEED = 42
random.seed(SEED)
EXPERIMENT_ID = "PIV-Q27-P0-v1.3"
GENERATION_CONTRACT_VERSION = "piv-q27-p0-agentic-v1.3"
PARSER_CONTRACT_VERSION = "piv-q27-toolxml-finalcode-v1.3"
PROMPT_CONTRACT_VERSION = "piv-q27-source-agentic-prompt-v1.3"

# -------------------------------------------------------------------------------------
# FROZEN SOURCE AUDIT
# -------------------------------------------------------------------------------------
SOURCE_FILE_SHA256 = {
    "qwen_exp1_base": "07d7c7a7ca4567a6118ae8cef00b10277aa37d5ada8698afb66afeb43c88aeb1",
    "qwen_runtime_benchmark": "7d91965196ec3309bf4e3f16f90efcaf2067ee8575d245c8eab9eb100f204b78",
    "arc2_advanced_eda": "1a9384d4b2e01fc15cddad95dee1d6eb5a633b6ae2d041d3d9ae97ffd1879f64",
    "nvarc_31_experiment_diagnostics": "69a64354b6b1c696af1f1c7a68984275d4db9b7eef5f2a382f879dc862d7c978",
    "portability_notes_v10": "c2dd4b45e3697d298d8f814b4087cee742ef4f8632f0eac318ba16cb8f8abda8",
    "experiment_master_plan": "8ca1c34c2c3d864c4b8e04fadf64465cbf468a09b3647c690f7306a07daf0c42"
}

PROXY48_TASK_IDS = [
    "0934a4d8",
    "20a9e565",
    "4c7dc4dd",
    "89565ca0",
    "136b0064",
    "291dc1e1",
    "16b78196",
    "271d71e2",
    "28a6681f",
    "2b83f449",
    "2c181942",
    "31f7f899",
    "35ab12c3",
    "3dc255db",
    "409aa875",
    "4c416de3",
    "64efde09",
    "6e453dd6",
    "7491f3cf",
    "7b0280bc",
    "7b80bb43",
    "7c66cb00",
    "1818057f",
    "332f06d7",
    "53fb4810",
    "7666fa5d",
    "da515329",
    "8f3a5a89",
    "db695cfb",
    "97d7923e",
    "de809cff",
    "c7f57c3e",
    "cb2d8a2c",
    "58490d8a",
    "dbff022c",
    "5961cc34",
    "80a900e0",
    "269e22fb",
    "8b9c3697",
    "e376de54",
    "b99e7126",
    "c4d067a0",
    "135a2760",
    "4a21e3da",
    "71e489b6",
    "800d221b",
    "dd6b8c4b",
    "36a08778"
]
PROXY48_ORDERED_SHA256 = hashlib.sha256("\n".join(PROXY48_TASK_IDS).encode("utf-8")).hexdigest()
PROXY48_EXPECTED_SHA256 = "8f432c60a35091dcda2b890574b1f4d3c03aaf48a5b3b7f96b4e88c35d359134"
PROXY48_SOURCE_COUNTS = {
    "total": 48,
    "tier1": 32,
    "tier2": 12,
    "tier3": 4,
    "easy_static": 16,
    "rare": 30,
    "multi_query": 8,
    "dynamic": 8,
    "hard_tier1": 2,
    "high_ratio": 5,
    "extreme_ratio": 4,
}

if len(PROXY48_TASK_IDS) != 48 or len(set(PROXY48_TASK_IDS)) != 48:
    raise RuntimeError("Frozen Proxy-48 must contain exactly 48 unique task IDs.")
if PROXY48_ORDERED_SHA256 != PROXY48_EXPECTED_SHA256:
    raise RuntimeError("Frozen Proxy-48 fingerprint mismatch. Refusing to run a changed suite.")

# NVARC notebook diagnostic anchors. These are SOURCE diagnostics only, never Q1 results.
NVARC_DIAGNOSTIC_ANCHORS = {
    "outputs_analyzed": 172,
    "gt_present_in_pool": 55,
    "gt_pool_coverage": 55/172,
    "gt_selected_by_current_dual": 45,
    "current_dual_score_reported": 33.50,
    "oracle_score_reported": 41.17,
    "avg_raw_duplicate_ratio": 0.378,
    "scorer_rank_correlation_on_gt_present": 0.961,
    "purpose": "Motivate complementary candidate coverage and provenance-aware dedup; not used to tune Q1 predictions.",
}


# =====================================================================================
# ONE-LINE RUN SWITCH
# =====================================================================================
RUN_PROFILE = "PREFLIGHT"   # "PREFLIGHT" | "SMOKE_GPU" | "PROXY48"
RESET_PROXY48_OUTPUTS = False

PIV_SOURCE_COMMIT = "b3d4494892c6d85fa2a35d548271fb02ea3e191b"
PIV_SOURCE_REPO = "https://github.com/gutfeeling/arc-agi-2-submission"
PIV_EXTERNAL_SOURCES = {
    "technical_article": "https://pivotools.github.io/pivotools-quarto-blog/posts/agentic_coding_arc_agi/",
    "audited_repo_snapshot": f"https://github.com/gutfeeling/arc-agi-2-submission/tree/{PIV_SOURCE_COMMIT}",
    "public_experiment_traces": "https://huggingface.co/datasets/arcagi2/arcagi2-agentic-coding-publication",
    "qwen_model_card": "https://huggingface.co/Qwen/Qwen3.8-27B-FP8",
    "qwen_chat_template": "https://huggingface.co/Qwen/Qwen3.8-27B-FP8/blob/main/chat_template.jinja",
}
PIV_SOURCE_FILES = {
    "clients.py": {"sha": "34a85ed34001923a5efae715f509e5aa7e8e235f", "role": "persistent model/tool loop + reasoning/tool history"},
    "solver.py": {"sha": "cf2ef554cc389322a5689988b300dc2bc05bf36a", "role": "one code turn -> fresh all-train verification -> test execution"},
    "turn.py": {"sha": "fe4dfc69c5963a56dfba84d0bd49821ae0d2e0bc", "role": "last code block + required solution() artifact"},
    "tools/repl_tool.py": {"sha": "691839335b1e65ba0bd2e4b8b84b70c50d559683", "role": "python tool, errors/timeouts as feedback, puzzle protection"},
    "utils/puzzle_utils.py": {"sha": "7e6ec3034e0dc2f94fba3aa9f1d1d4a0896120b4", "role": "all-train/all-test-input serialization; test outputs hidden"},
    "utils/solving_utils.py": {"sha": "197be842b3d6fa08e4e98066e801dc8cf2bc8327", "role": "fresh sandbox train check and test execution"},
    "utils/code_templates/check_solution_on_training_examples.py": {"sha": "c13ea5b89360ea997a30a1eabe2ccb4a67489b44", "role": "deep-copy input + exact train equality"},
    "solver/prompts/agentic_coding_solver.txt": {"sha": "dab87e08bbe3f071920f13eb98f4a4d6f03ada10", "role": "ARC priors, preloaded puzzle, self-contained solution()"},
    "solver/prompts/system_prompt.txt": {"sha": "265131a50fb42deebe0a8da353f466b11d6d7d6b", "role": "stateful Python tool contract"},
    "solver/config/gpt_oss_120b_high.py": {"sha": "8fd563ddcb10962c7564fbfa8cbf3424a7a437d7", "role": "source agentic config example"},
    "sandbox/ipybox_sandbox.py": {"sha": "195e0e66ad7328a4f823974f04778b639a16412a", "role": "source Docker/IPyBox persistent REPL implementation; semantics only, not ported literally"},
    "sandbox/base.py": {"sha": "517f5794ff0cac64024e47c184d116b1f36577ea", "role": "source stateful REPL/Sandbox interface and fresh run_cells semantics"},
    "evaluation/evaluate.py": {"sha": "8b48fb4482dc7d9772ac010ba6dd81f6cc15e0a4", "role": "source multi-sample evaluation / majority aggregation; not copied into P0"},
    "utils/code_templates/produce_output_grid.py": {"sha": "b6ab435ca4f7d77414764e3a4400bb8c6dbead5e", "role": "execute verified solution on all test inputs"},
}

PROFILE_TABLE = {
    "PREFLIGHT": {
        "use_model": False, "task_set": "PROXY48", "active_tasks": 48,
        "agent_batch_size": 8, "max_tool_calls": 6, "max_model_turns": 8,
        "per_turn_max_tokens": 4096, "total_generated_token_cap": 12288,
        "max_retries": 2,
        "tool_timeout_sec": 10.0, "verifier_timeout_sec": 20.0,
        "bundle_name": "PIV_Q27_P0_PREFLIGHT_BUNDLE.zip",
        "reset_outputs": True, "accuracy_is_real": False,
    },
    "SMOKE_GPU": {
        "use_model": True, "task_set": "SMOKE8", "active_tasks": 8,
        "agent_batch_size": 8, "max_tool_calls": 2, "max_model_turns": 4,
        "per_turn_max_tokens": 2048, "total_generated_token_cap": 6144,
        "max_retries": 2,
        "tool_timeout_sec": 10.0, "verifier_timeout_sec": 20.0,
        "bundle_name": "PIV_Q27_P0_SMOKE_GPU_BUNDLE.zip",
        "reset_outputs": True, "accuracy_is_real": False,
    },
    "PROXY48": {
        "use_model": True, "task_set": "PROXY48", "active_tasks": 48,
        "agent_batch_size": 8, "max_tool_calls": 6, "max_model_turns": 8,
        "per_turn_max_tokens": 4096, "total_generated_token_cap": 12288,
        "max_retries": 2,
        "tool_timeout_sec": 10.0, "verifier_timeout_sec": 20.0,
        "bundle_name": "PIV_Q27_P0_PROXY48_BUNDLE.zip",
        "reset_outputs": RESET_PROXY48_OUTPUTS, "accuracy_is_real": True,
    },
}
RUN_PROFILE = str(RUN_PROFILE).strip().upper()
if RUN_PROFILE not in PROFILE_TABLE:
    raise ValueError(f"RUN_PROFILE={RUN_PROFILE!r} invalid; choose {list(PROFILE_TABLE)}")
PROFILE = PROFILE_TABLE[RUN_PROFILE]
USE_MODEL = bool(PROFILE["use_model"])
TASK_SET = str(PROFILE["task_set"])
ACTIVE_TASK_LIMIT = int(PROFILE["active_tasks"])
AGENT_BATCH_SIZE = int(PROFILE["agent_batch_size"])
MAX_TOOL_CALLS = int(PROFILE["max_tool_calls"])
MAX_MODEL_TURNS = int(PROFILE["max_model_turns"])
MAX_RETRIES = int(PROFILE.get("max_retries", 2))
PER_TURN_MAX_TOKENS = int(PROFILE["per_turn_max_tokens"])
TOTAL_GENERATED_TOKEN_CAP = int(PROFILE["total_generated_token_cap"])
TOOL_TIMEOUT_SEC = float(PROFILE["tool_timeout_sec"])
VERIFIER_TIMEOUT_SEC = float(PROFILE["verifier_timeout_sec"])
BUNDLE_NAME = str(PROFILE["bundle_name"])
ACCURACY_IS_REAL = bool(PROFILE["accuracy_is_real"])

MODEL_PATH = Path("/kaggle/input/models/foysalemonshanto/qwen3-8-27b-fp8-repacked-v1/pytorch/hf-fp8/1")
WHEELHOUSE = Path("/kaggle/input/datasets/driessmit1/arc3-vllm-h100-wheelhouse-v3")
ARC_ROOT = Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-2")
TRAIN_CHALLENGES_PATH = ARC_ROOT / "arc-agi_training_challenges.json"
TRAIN_SOLUTIONS_PATH  = ARC_ROOT / "arc-agi_training_solutions.json"
EVAL_CHALLENGES_PATH  = ARC_ROOT / "arc-agi_evaluation_challenges.json"
EVAL_SOLUTIONS_PATH   = ARC_ROOT / "arc-agi_evaluation_solutions.json"
TEST_CHALLENGES_PATH  = ARC_ROOT / "arc-agi_test_challenges.json"
SAMPLE_SUBMISSION_PATH = ARC_ROOT / "sample_submission.json"

BASE_WORK_ROOT = Path("/kaggle/working/piv_q27_p0_v13")
WORK_ROOT = BASE_WORK_ROOT / RUN_PROFILE.lower()
if PROFILE["reset_outputs"] and WORK_ROOT.exists():
    shutil.rmtree(WORK_ROOT)
WORK_ROOT.mkdir(parents=True, exist_ok=True)
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
NOTEBOOK_WALL_START = time.perf_counter()
TOTAL_RUNTIME_SEC = 12 * 60 * 60
SAFETY_RESERVE_SEC = 30 * 60

# Frozen from the successful Q1 4×L4 runtime.
MAX_MODEL_LEN = 29696
MAX_NUM_BATCHED_TOKENS = 32768
MAX_NUM_SEQS = 16
GPU_MEMORY_UTILIZATION = 0.90

# EXP-0 diagnostics preserved unchanged.
WEIGHT_CLIP_LOW = 0.20
WEIGHT_CLIP_HIGH = 5.00
N_PROPENSITY_FOLDS = 5
HIGH_RATIO_THRESHOLD = 10.0
LARGE_TEST_INPUT_CELLS = 300
MANY_COLORS_THRESHOLD = 7
LOW_BG_THRESHOLD = 0.30

os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2,3"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

print("=" * 116)
print(EXPERIMENT_ID, "| Stateful Python REPL + Fresh Exact Train Verifier")
print("=" * 116)
print("RUN_ID:", RUN_ID, "| RUN_PROFILE:", RUN_PROFILE, "| TASK_SET:", TASK_SET)
print("PIV source commit:", PIV_SOURCE_COMMIT)
print("Agent budget: calls/turns/per-turn/total =", MAX_TOOL_CALLS, MAX_MODEL_TURNS, PER_TURN_MAX_TOKENS, TOTAL_GENERATED_TOKEN_CAP)
print("Proxy-48 SHA256:", PROXY48_ORDERED_SHA256)
print("WORK_ROOT:", WORK_ROOT)

required = [MODEL_PATH, WHEELHOUSE, ARC_ROOT, TRAIN_CHALLENGES_PATH, TRAIN_SOLUTIONS_PATH,
            EVAL_CHALLENGES_PATH, EVAL_SOLUTIONS_PATH, TEST_CHALLENGES_PATH, SAMPLE_SUBMISSION_PATH]
for p in required:
    if not p.exists():
        raise FileNotFoundError(f"Required Kaggle input is missing: {p}")
config_path = MODEL_PATH / "config.json"
index_path = MODEL_PATH / "model.safetensors.index.json"
with open(config_path, "r", encoding="utf-8") as f:
    model_cfg = json.load(f)
qcfg = model_cfg.get("quantization_config", {})
if model_cfg.get("model_type") != "qwen3_5":
    raise RuntimeError(f"Unexpected model_type={model_cfg.get('model_type')}")
if qcfg.get("quant_method") != "fp8" or qcfg.get("fmt") != "e4m3":
    raise RuntimeError(f"Unexpected quantization_config={qcfg}")

def pkg_version(name):
    try: return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError: return None

def stable_json_sha256(obj):
    payload = json.dumps(obj, sort_keys=True, separators=(",", ":"), default=str, ensure_ascii=False)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()

def small_file_sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

EXPECTED_RUNTIME_PACKAGES = {"vllm":"0.19.0", "transformers":"4.57.6", "tokenizers":"0.22.2", "huggingface-hub":"0.36.2"}
_loaded_module_names = {"vllm":"vllm", "transformers":"transformers", "tokenizers":"tokenizers", "huggingface-hub":"huggingface_hub"}
before_stack = {name: pkg_version(name) for name in EXPECTED_RUNTIME_PACKAGES}
for _pkg, _module in _loaded_module_names.items():
    if before_stack.get(_pkg) != EXPECTED_RUNTIME_PACKAGES[_pkg] and _module in sys.modules:
        raise RuntimeError(f"{_pkg} already imported at {before_stack.get(_pkg)!r}; restart Kaggle session before Run All.")

def offline_pip_install(requirements):
    if not requirements: return
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--no-index", "--no-cache-dir",
                           "--find-links", str(WHEELHOUSE), *requirements])
if before_stack["vllm"] != EXPECTED_RUNTIME_PACKAGES["vllm"]:
    offline_pip_install(["vllm==0.19.0"])
else:
    repair=[]
    for package in ["transformers","tokenizers","huggingface-hub"]:
        if before_stack[package] != EXPECTED_RUNTIME_PACKAGES[package]:
            module_name = package.replace("-", "_")
            if module_name in sys.modules or (package=="huggingface-hub" and "huggingface_hub" in sys.modules):
                raise RuntimeError(f"{package} already imported with wrong version; restart session.")
            repair.append(f"{package}=={EXPECTED_RUNTIME_PACKAGES[package]}")
    offline_pip_install(repair)
after_stack = {name: pkg_version(name) for name in EXPECTED_RUNTIME_PACKAGES}
for package, wanted in EXPECTED_RUNTIME_PACKAGES.items():
    if after_stack.get(package) != wanted:
        raise RuntimeError(f"Runtime parity failure: {package}={after_stack.get(package)!r}, expected {wanted!r}")
print("Runtime dependency parity PASS:", after_stack)


In [ ]:

# =====================================================================================
# CELL 2 — LOAD ARC DATA
# =====================================================================================
import numpy as np
import pandas as pd
from IPython.display import display

np.random.seed(SEED)

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

train_challenges = load_json(TRAIN_CHALLENGES_PATH)
train_solutions  = load_json(TRAIN_SOLUTIONS_PATH)
eval_challenges  = load_json(EVAL_CHALLENGES_PATH)
eval_solutions   = load_json(EVAL_SOLUTIONS_PATH)
test_challenges  = load_json(TEST_CHALLENGES_PATH)
sample_submission = load_json(SAMPLE_SUBMISSION_PATH)

assert set(eval_challenges) == set(eval_solutions)
assert set(test_challenges) == set(sample_submission)
missing_proxy_ids = [k for k in PROXY48_TASK_IDS if k not in eval_challenges or k not in eval_solutions]
if missing_proxy_ids:
    raise RuntimeError(f"Frozen Proxy-48 IDs missing from public Evaluation data: {missing_proxy_ids}")

print("Training tasks       :", len(train_challenges))
print("Evaluation tasks     :", len(eval_challenges))
print("Evaluation outputs   :", sum(len(v) for v in eval_solutions.values()))
print("Test tasks           :", len(test_challenges))
print("Visible test queries :", sum(len(v["test"]) for v in test_challenges.values()))


In [ ]:

# =====================================================================================
# CELL 3 — EXP-0 STRUCTURAL FEATURES + TEST-RELEVANCE WEIGHTS
# Recomputed here so EXP-1 is self-contained.
# =====================================================================================
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score, log_loss

def arr(grid):
    return np.asarray(grid, dtype=np.int16)

def is_rectangular_grid(grid, max_hw=30):
    if not isinstance(grid, list) or len(grid) < 1 or len(grid) > max_hw:
        return False
    if not all(isinstance(row, list) for row in grid):
        return False
    widths = [len(row) for row in grid]
    if min(widths, default=0) < 1 or max(widths, default=0) > max_hw or len(set(widths)) != 1:
        return False
    for row in grid:
        for x in row:
            if isinstance(x, bool) or not isinstance(x, (int, np.integer)):
                return False
            if int(x) < 0 or int(x) > 9:
                return False
    return True

def grid_basic_stats(grid):
    a = arr(grid)
    vals, counts = np.unique(a, return_counts=True)
    probs = counts / counts.sum()
    entropy = float(-(probs * np.log(probs + 1e-12)).sum())
    return {
        "h": int(a.shape[0]),
        "w": int(a.shape[1]),
        "cells": int(a.size),
        "n_colors": int(len(vals)),
        "bg0_density": float(np.mean(a == 0)),
        "color_entropy": entropy,
    }

def colored_components_count(grid):
    # Diagnostic only: 4-connected same-color non-zero components.
    a = arr(grid)
    h, w = a.shape
    seen = np.zeros((h, w), dtype=bool)
    n = 0
    sizes = []
    for r in range(h):
        for c in range(w):
            if seen[r, c] or a[r, c] == 0:
                continue
            color = a[r, c]
            q = deque([(r, c)])
            seen[r, c] = True
            size = 0
            while q:
                rr, cc = q.popleft()
                size += 1
                for dr, dc in ((1,0),(-1,0),(0,1),(0,-1)):
                    nr, nc = rr+dr, cc+dc
                    if 0 <= nr < h and 0 <= nc < w and not seen[nr, nc] and a[nr, nc] == color:
                        seen[nr, nc] = True
                        q.append((nr, nc))
            n += 1
            sizes.append(size)
    return n, (float(np.mean(sizes)) if sizes else 0.0)

def summarize(values, prefix):
    x = np.asarray(values, dtype=float)
    if len(x) == 0:
        return {
            f"{prefix}_mean": 0.0, f"{prefix}_std": 0.0,
            f"{prefix}_min": 0.0, f"{prefix}_max": 0.0,
        }
    return {
        f"{prefix}_mean": float(x.mean()),
        f"{prefix}_std": float(x.std()),
        f"{prefix}_min": float(x.min()),
        f"{prefix}_max": float(x.max()),
    }

def task_features(task_id, task, split):
    train = task["train"]
    tests = task["test"]

    tr_in_stats, tr_out_stats, te_in_stats = [], [], []
    tr_in_components, tr_out_components, te_components = [], [], []
    tr_in_component_sizes, tr_out_component_sizes, te_component_sizes = [], [], []
    ratios, h_ratios, w_ratios, same_shape = [], [], [], []
    all_visible_colors = set()

    for ex in train:
        si = grid_basic_stats(ex["input"])
        so = grid_basic_stats(ex["output"])
        tr_in_stats.append(si); tr_out_stats.append(so)
        ci, csi = colored_components_count(ex["input"])
        co, cso = colored_components_count(ex["output"])
        tr_in_components.append(ci); tr_out_components.append(co)
        tr_in_component_sizes.append(csi); tr_out_component_sizes.append(cso)
        ratios.append(so["cells"] / max(si["cells"], 1))
        h_ratios.append(so["h"] / max(si["h"], 1))
        w_ratios.append(so["w"] / max(si["w"], 1))
        same_shape.append(int((si["h"], si["w"]) == (so["h"], so["w"])))
        all_visible_colors.update(np.unique(arr(ex["input"])).tolist())
        all_visible_colors.update(np.unique(arr(ex["output"])).tolist())

    for ex in tests:
        st = grid_basic_stats(ex["input"])
        te_in_stats.append(st)
        ct, cst = colored_components_count(ex["input"])
        te_components.append(ct); te_component_sizes.append(cst)
        all_visible_colors.update(np.unique(arr(ex["input"])).tolist())

    feat = {
        "task_id": task_id,
        "split": split,
        "n_train": len(train),
        "n_test_queries": len(tests),
        "visible_n_colors": len(all_visible_colors),
        "same_shape_frac": float(np.mean(same_shape)),
        "dynamic_any": int(any(x == 0 for x in same_shape)),
        "multi_query": int(len(tests) > 1),
    }

    for key in ["h", "w", "cells", "n_colors", "bg0_density", "color_entropy"]:
        feat.update(summarize([x[key] for x in tr_in_stats], f"tr_in_{key}"))
        feat.update(summarize([x[key] for x in tr_out_stats], f"tr_out_{key}"))
        feat.update(summarize([x[key] for x in te_in_stats], f"te_in_{key}"))

    feat.update(summarize(ratios, "area_ratio"))
    feat.update(summarize(h_ratios, "h_ratio"))
    feat.update(summarize(w_ratios, "w_ratio"))
    feat.update(summarize(tr_in_components, "tr_in_components"))
    feat.update(summarize(tr_out_components, "tr_out_components"))
    feat.update(summarize(te_components, "te_in_components"))
    feat.update(summarize(tr_in_component_sizes, "tr_in_component_size"))
    feat.update(summarize(tr_out_component_sizes, "tr_out_component_size"))
    feat.update(summarize(te_component_sizes, "te_in_component_size"))

    feat["n_train_eq_2"] = int(len(train) == 2)
    feat["n_train_ge_3"] = int(len(train) >= 3)
    feat["high_ratio"] = int(max(ratios) >= HIGH_RATIO_THRESHOLD)
    feat["large_test_input"] = int(max(x["cells"] for x in te_in_stats) >= LARGE_TEST_INPUT_CELLS)
    feat["many_colors"] = int(len(all_visible_colors) >= MANY_COLORS_THRESHOLD)
    feat["low_bg_train_input"] = int(
        np.mean([x["bg0_density"] for x in tr_in_stats]) < LOW_BG_THRESHOLD
    )
    return feat

def build_feature_df(data, split):
    return pd.DataFrame(
        [task_features(k, v, split) for k, v in data.items()]
    ).sort_values("task_id").reset_index(drop=True)

train_feat = build_feature_df(train_challenges, "train")
eval_feat  = build_feature_df(eval_challenges, "eval")
test_feat  = build_feature_df(test_challenges, "test")

for name, df in [("train", train_feat), ("eval", eval_feat), ("test", test_feat)]:
    df.to_csv(WORK_ROOT / f"task_features_{name}.csv", index=False)

numeric_cols = [
    c for c in eval_feat.columns
    if c not in {"task_id", "split"} and pd.api.types.is_numeric_dtype(eval_feat[c])
]
EXCLUDE_PROPENSITY = {
    "task_id", "split",
    "n_train_eq_2", "n_train_ge_3", "high_ratio", "large_test_input",
    "many_colors", "low_bg_train_input", "multi_query",
}
prop_cols = [c for c in numeric_cols if c not in EXCLUDE_PROPENSITY]

combined = pd.concat(
    [eval_feat.assign(_is_test=0), test_feat.assign(_is_test=1)],
    ignore_index=True
)
X = combined[prop_cols].astype(float)
y = combined["_is_test"].astype(int).to_numpy()

prop_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
    ("clf", LogisticRegression(
        C=0.5, penalty="l2", solver="liblinear",
        max_iter=5000, random_state=SEED
    )),
])
cv = StratifiedKFold(n_splits=N_PROPENSITY_FOLDS, shuffle=True, random_state=SEED)
p_oof = cross_val_predict(prop_model, X, y, cv=cv, method="predict_proba")[:, 1]

prop_auc = roc_auc_score(y, p_oof)
prop_logloss = log_loss(y, p_oof)

n_eval = len(eval_feat)
n_test = len(test_feat)
prior_eval = n_eval / (n_eval + n_test)
prior_test = n_test / (n_eval + n_test)

p_eval_as_test = np.clip(p_oof[:n_eval], 1e-4, 1 - 1e-4)
density_raw = (p_eval_as_test / (1 - p_eval_as_test)) * (prior_eval / prior_test)
density_clip = np.clip(density_raw, WEIGHT_CLIP_LOW, WEIGHT_CLIP_HIGH)
density_norm = density_clip / np.mean(density_clip)

eval_weights = eval_feat[["task_id"]].copy()
eval_weights["p_test_oof"] = p_eval_as_test
eval_weights["density_ratio_raw"] = density_raw
eval_weights["weight"] = density_norm
eval_weights["test_relevance_rank_pct"] = pd.Series(density_norm).rank(pct=True).to_numpy()
eval_weights["test_relevance_quartile"] = pd.qcut(
    pd.Series(density_norm).rank(method="first"),
    q=4,
    labels=["Q1_low", "Q2", "Q3", "Q4_high"]
).astype(str).to_numpy()

weight_ess = float((density_norm.sum() ** 2) / np.sum(density_norm ** 2))
eval_weights.to_csv(WORK_ROOT / "eval_test_propensity_weights.csv", index=False)

print("EXP-0 reproduction:")
print("  Eval-vs-Test CV AUC :", round(prop_auc, 6))
print("  Log loss            :", round(prop_logloss, 6))
print("  Weight ESS          :", round(weight_ess, 3), "/", len(eval_weights))
print("  Weight min/median/max:",
      round(float(density_norm.min()), 4),
      round(float(np.median(density_norm)), 4),
      round(float(density_norm.max()), 4))


# =====================================================================================
# SOURCE-ALIGNED PROXY-48 FEATURES
# Exact feature/tag rules copied from the source EDA notebook for audit comparability.
# These DO NOT replace EXP-0 features or official scoring.
# =====================================================================================
def source_eda_n_cells(grid):
    return int(np.prod(np.shape(grid)))

def source_eda_proxy_features(task_id, data):
    puzzle = data[task_id]
    train = puzzle["train"]
    input_sizes = [source_eda_n_cells(ex["input"]) for ex in train]
    output_sizes = [source_eda_n_cells(ex["output"]) for ex in train]
    ratios = [max(i, o) / max(min(i, o), 1) for i, o in zip(input_sizes, output_sizes)]
    colors = set()
    pixels = []
    for ex in train:
        for field in ["input", "output"]:
            a = np.asarray(ex[field])
            colors.update(np.unique(a).tolist())
            pixels.extend(a.reshape(-1).tolist())

    avg_input_size = float(np.mean(input_sizes))
    avg_ratio = float(np.mean(ratios))
    max_ratio = float(max(ratios))
    n_colors = len(colors)
    n_train = len(train)
    n_queries = len(puzzle["test"])
    is_dynamic = any(np.shape(ex["input"]) != np.shape(ex["output"]) for ex in train)
    is_rare = bool(6 in colors or 9 in colors)
    bg_density = pixels.count(0) / max(len(pixels), 1)

    if is_dynamic and max_ratio > 30:
        tier = 3
    elif is_dynamic and max_ratio > 5:
        tier = 2
    elif avg_input_size > 400:
        tier = 2
    else:
        tier = 1

    fine = (
        avg_input_size / 900 * 0.30 +
        (avg_ratio - 1) / 10 * 0.25 +
        n_colors / 10 * 0.20 +
        (0.25 if is_dynamic else 0.0)
    )
    if n_train >= 6 and tier < 3:
        fine *= 0.7

    tags = [f"tier{tier}"]
    if tier == 1 and (not is_dynamic) and max_ratio == 1 and n_colors <= 5:
        tags.append("easy_static")
    if tier == 1 and fine <= 0.22:
        tags.append("easy")
    if tier == 1 and fine > 0.45:
        tags.append("hard_tier1")
    if is_dynamic:
        tags.append("dynamic")
    if is_rare:
        tags.append("rare")
    if n_queries > 1:
        tags.append("multi_query")
    if max_ratio > 5:
        tags.append("high_ratio")
    if max_ratio > 30:
        tags.append("extreme_ratio")
    if n_colors >= 8:
        tags.append("many_colors")
    if avg_input_size >= 500:
        tags.append("large")
    if bg_density >= 0.70:
        tags.append("sparse_bg")
    if bg_density <= 0.30:
        tags.append("dense_non_bg")
    if n_train <= 2:
        tags.append("low_train")

    return {
        "task_id": task_id,
        "proxy_tier": int(tier),
        "proxy_fine": float(fine),
        "proxy_avg_input_size": avg_input_size,
        "proxy_max_ratio": max_ratio,
        "proxy_avg_ratio": avg_ratio,
        "proxy_n_colors": int(n_colors),
        "proxy_dynamic": int(is_dynamic),
        "proxy_rare": int(is_rare),
        "proxy_n_train": int(n_train),
        "proxy_n_queries": int(n_queries),
        "proxy_bg_density": float(bg_density),
        "proxy_easy_static": int("easy_static" in tags),
        "proxy_hard_tier1": int("hard_tier1" in tags),
        "proxy_high_ratio": int("high_ratio" in tags),
        "proxy_extreme_ratio": int("extreme_ratio" in tags),
        "proxy_tags": "|".join(tags),
    }

proxy_source_feat = pd.DataFrame([
    source_eda_proxy_features(k, eval_challenges) for k in eval_challenges
]).sort_values("task_id").reset_index(drop=True)
proxy_source_feat.to_csv(WORK_ROOT / "source_eda_features_eval.csv", index=False)

proxy48_source = proxy_source_feat[proxy_source_feat["task_id"].isin(PROXY48_TASK_IDS)].copy()
proxy48_order = {k: i for i, k in enumerate(PROXY48_TASK_IDS)}
proxy48_source["_order"] = proxy48_source["task_id"].map(proxy48_order)
proxy48_source = proxy48_source.sort_values("_order").drop(columns="_order").reset_index(drop=True)
proxy48_source.to_csv(WORK_ROOT / "proxy48_source_eda.csv", index=False)

proxy_source_counts = {
    "total": int(len(proxy48_source)),
    "tier1": int((proxy48_source["proxy_tier"] == 1).sum()),
    "tier2": int((proxy48_source["proxy_tier"] == 2).sum()),
    "tier3": int((proxy48_source["proxy_tier"] == 3).sum()),
    "easy_static": int(proxy48_source["proxy_easy_static"].sum()),
    "rare": int(proxy48_source["proxy_rare"].sum()),
    "multi_query": int((proxy48_source["proxy_n_queries"] > 1).sum()),
    "dynamic": int(proxy48_source["proxy_dynamic"].sum()),
    "hard_tier1": int(proxy48_source["proxy_hard_tier1"].sum()),
    "high_ratio": int(proxy48_source["proxy_high_ratio"].sum()),
    "extreme_ratio": int(proxy48_source["proxy_extreme_ratio"].sum()),
}
if proxy_source_counts != PROXY48_SOURCE_COUNTS:
    raise RuntimeError(
        "Proxy-48 source-EDA profile drift detected. "
        f"expected={PROXY48_SOURCE_COUNTS}, actual={proxy_source_counts}"
    )
with open(WORK_ROOT / "proxy48_source_distribution.json", "w", encoding="utf-8") as f:
    json.dump(proxy_source_counts, f, indent=2)
print("Proxy-48 source-EDA contract: PASS", proxy_source_counts)


In [ ]:
# =====================================================================================
# CELL 4 — TOKENIZER + SOURCE-ALIGNED AGENTIC PROMPT + QWEN TOOL TEMPLATE + ACTIVE TASKS
# =====================================================================================
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(str(MODEL_PATH), local_files_only=True, trust_remote_code=False)
print("Tokenizer:", tokenizer.__class__.__name__, "| vocab:", len(tokenizer))

# OpenAI-style schema; Qwen3.8's official chat template serializes it into its XML tool-call contract.
PYTHON_TOOL = {
    "type": "function",
    "function": {
        "name": "python",
        "description": "Execute Python in a persistent per-puzzle REPL and return stdout or a concise error.",
        "parameters": {
            "type": "object",
            "properties": {"code": {"type": "string", "description": "Python code to execute"}},
            "required": ["code"],
            "additionalProperties": False,
        },
    },
}
TOOL_SCHEMA_SHA256 = stable_json_sha256(PYTHON_TOOL)

SOURCE_SYSTEM_PROMPT = '# Tools\n\n## python\n\nUse this tool to execute Python code in your chain of thought. \n\nWhen you send a message containing Python code to python, it will be executed in a stateful Jupyter notebook environment. The environment supports common third party libraries like numpy, scipy, shapely, networkx, scikit-image, more-itertools, pillow, python-constraint, ortools and z3-solver. python will respond with the output of the execution or time out after 120.0 seconds. \n\n### Usage guideline\n\n1. Use the variables and functions already loaded in the Jupyter notebook instead of defining them yourself. The user message will provide a complete list of such variables and functions.\n2. Prefer to write reusable functions. These will persist in the stateful Jupyter notebook and can be referenced in later code snippets without having to define them again.'
SOURCE_AGENTIC_PROMPT_TEMPLATE = 'You are an expert ARC AGI puzzle solver. Please solve the puzzle given below.\n\n# Background information about ARC AGI puzzles\n\nEach task consists of a small number of demonstration examples (3.3 on average), and a small number of test examples (1 - 3).\n\nEach example consists of an “input grid” and an “output grid”. Each “grid” is a literal grid of colored squares There are 10 unique colors. A grid can be any height or width between 1x1 and 30x30, inclusive.\n\nThe color code is as follows:\n\n0 - black\n1 - navy blue\n2 - red\n3 - green\n4 - yellow\n5 - gray\n6 - pink\n7 - orange\n8 - sky blue\n9 - maroon\n\nWhen solving an evaluation task, a test-taker has access to the training examples for the task (both the input and output grids), as well as the input grid of the test examples for the task. The test-taker must construct on its own the output grid corresponding to the input grid of each test example. “Constructing the output grid” is done entirely from scratch, meaning that the test-taker must decide what the height and width of the output grid should be, what colors they should place on the grid, and where. The task is successfully solved if the test-taker can produce the exact correct answer on all test examples for the task (binary measure of success).\n\nThe mapping from integers (0-9) to color names is arbitrary and should be treated as categorical labels; the transformation rule never relies on the specific integer–color pairing.\n\nARC AGI puzzle solutions feel "natural" to humans. Natural rules tend to build on visual priors and small-number counting. They are visually immediate or prominent for humans. That is why humans from all educational backgrounds are able to solve these puzzles. Rules that depend on non-visual considerations (e.g. mathematics that\'s more advanced than small number counting) or inelegant mechanics (e.g. tie-breakers, complex conditional rules, special exceptions) are considered "unnatural."  \n\nThis doesn\'t mean puzzles have to be simple or easy. Common patterns that puzzles use to increase difficulty are:  \n\n- **Rule composition:** multiple simple, natural rules are applied in sequence, with one transformation setting up the next.\n- **Contextual application:** a simple if/else based on a visually clear feature.\n- **In-grid symbol definition:** certain shapes or patterns act as symbols whose meaning is established using the training examples and which control some aspect of the transformation. The meaning can even vary across training examples.\n- **Iterated steps:** a simple action is applied repeatedly (as in assembling pieces of a jigsaw), where each step constrains the next. Repetition is restricted to a small number.\n- **Approximate features**: Features that are easily detected by humans, but hard to detect using naive algorithms. Examples are slightly broken symmetry, slightly broken continuity, off-center symmetry axes, objects defined via 8-connectivity etc.\n- **Non-locality:** the solution requires applying changes to one region, with the changes depending on features in a different, distant region.\n\n# Puzzle to solve\n\n{{puzzle}}\n\n# Task aids\n\nYou may execute code in the Jupyter notebook environment.\n\nVariables available in the Jupyter notebook:\n\n- `puzzle`: A Python `dict` containing the training and test grids. The key "train" contains the training examples in a list. Similarly, the key "test" contains the test examples as a list. Each training/test example is a dict containing keys "input" and "output". The values are the grids expressed as list of lists of integers (0 - 9). Access: `puzzle["train"][i]["input"]`, `puzzle["train"][i]["output"]`, `puzzle["test"][j]["input"]`.\n\nYour very first code execution should be `puzzle.keys()` to check data access. \n\nDon\'t print the `puzzle` dict, because it prints out the grids that are already present in this user message and consumes valuable token budget.\n\n# Your task\n\nPlease solve the puzzle. Solving the puzzle means constructing a function `solution(grid: list[list[int]])` that implements the transformation from input grid to output grid shown in the training examples. The solution must work on all training examples.\n\n```\nfor idx, example in enumerate(puzzle["train"]):\n    assert solution(example["input"]) == example["output"], f"Train example {idx}: mismatch"\n```\n\nThe solution must generalize to the test input(s). Solutions that cheat by hardcoding the training example outputs are not acceptable.\n\n# Output format (STRICT)\n\nYour response must contain a single Python code block fenced by ```python ... ```. Must include a `solution` function. Must be self-contained i.e. include all necessary imports and helper functions.'

# LOCAL adaptation only: source advertises a 120s IPython timeout; P0 intentionally starts at 10s.
SYSTEM_PROMPT = SOURCE_SYSTEM_PROMPT.replace(
    "time out after 120.0 seconds",
    f"time out after {TOOL_TIMEOUT_SEC:.1f} seconds",
)
SYSTEM_PROMPT += (
    f"\n\n### Local P0 compute budget\n"
    f"You may execute Python at most {MAX_TOOL_CALLS} times in this task. "
    "Use the calls deliberately. When the final Python call has been used, do not request another tool call; "
    "return the self-contained final `solution(grid)` code block on the next assistant turn."
)

def grid_lines(grid):
    return "\n".join(" ".join(str(int(x)) for x in row) for row in grid)

def puzzle_to_source_text(task):
    parts = ["## Training examples"]
    for i, ex in enumerate(task["train"]):
        parts += [f"### Training example {i}", "Input:", grid_lines(ex["input"]), "Output:", grid_lines(ex["output"])]
    parts.append("## Tests" if len(task["test"]) > 1 else "## Test")
    for i, ex in enumerate(task["test"]):
        if len(task["test"]) > 1:
            parts.append(f"### Test {i}")
        parts += ["Input:", grid_lines(ex["input"])]
    return "\n\n".join(parts)

def build_agent_user_prompt(task):
    # Keep the source prompt intact except for two necessary local substitutions:
    # 1) serialize the current puzzle; 2) clarify that test outputs are absent from the preloaded variable.
    prompt = SOURCE_AGENTIC_PROMPT_TEMPLATE.replace("{{puzzle}}", puzzle_to_source_text(task))
    prompt = prompt.replace(
        'Each training/test example is a dict containing keys "input" and "output". '
        'The values are the grids expressed as list of lists of integers (0 - 9). '
        'Access: `puzzle["train"][i]["input"]`, `puzzle["train"][i]["output"]`, `puzzle["test"][j]["input"]`.',
        'Each training example is a dict containing keys "input" and "output"; each test example contains only "input". '
        'The values are grids expressed as list of lists of integers (0 - 9). '
        'Access: `puzzle["train"][i]["input"]`, `puzzle["train"][i]["output"]`, `puzzle["test"][j]["input"]`.'
    )
    return prompt

SOURCE_AGENTIC_PROMPT_LOCAL_SHA256 = hashlib.sha256(SOURCE_AGENTIC_PROMPT_TEMPLATE.encode("utf-8")).hexdigest()
SOURCE_SYSTEM_PROMPT_LOCAL_SHA256 = hashlib.sha256(SOURCE_SYSTEM_PROMPT.encode("utf-8")).hexdigest()

def render_agent_chat(messages):
    kwargs = dict(
        tokenize=False,
        add_generation_prompt=True,
        tools=[PYTHON_TOOL],
        enable_thinking=True,
        reasoning_effort="xhigh",
        preserve_thinking=True,
    )
    try:
        return tokenizer.apply_chat_template(messages, **kwargs), "qwen_tools+xhigh+preserve_thinking"
    except TypeError as exc:
        # Preserve the causal mechanism if this repacked tokenizer predates the explicit preserve kwarg.
        kwargs.pop("preserve_thinking", None)
        rendered = tokenizer.apply_chat_template(messages, **kwargs)
        return rendered, f"qwen_tools+xhigh(template-preserve-default); fallback={type(exc).__name__}"

# Frozen task selection: same Proxy-48; SMOKE is a deterministic subset of it.
def choose_smoke8_from_proxy():
    f = proxy_source_feat.set_index("task_id").loc[PROXY48_TASK_IDS].copy()
    chosen=[]
    def add(x):
        if x not in chosen: chosen.append(x)
    # Deliberately span easy/hard/dynamic/multi-query/high-ratio while remaining frozen within Proxy48.
    for cond in [
        lambda d: d["proxy_tier"]==3,
        lambda d: d["proxy_dynamic"]==1,
        lambda d: d["proxy_n_queries"]>1,
        lambda d: d["proxy_high_ratio"]==1,
        lambda d: d["proxy_easy_static"]==1,
        lambda d: d["proxy_rare"]==1,
    ]:
        sub=f[cond(f)]
        if len(sub): add(sub.index[0])
    for x in PROXY48_TASK_IDS:
        add(x)
        if len(chosen)>=8: break
    return chosen[:8]

if TASK_SET == "PROXY48": ACTIVE_TASK_IDS = list(PROXY48_TASK_IDS)
elif TASK_SET == "SMOKE8": ACTIVE_TASK_IDS = choose_smoke8_from_proxy()
else: raise RuntimeError(f"Unknown TASK_SET={TASK_SET}")
ACTIVE_TASK_SET=set(ACTIVE_TASK_IDS)
if len(ACTIVE_TASK_IDS)!=ACTIVE_TASK_LIMIT or len(ACTIVE_TASK_SET)!=ACTIVE_TASK_LIMIT:
    raise RuntimeError("Active task count/uniqueness mismatch")
if not ACTIVE_TASK_SET.issubset(set(PROXY48_TASK_IDS)):
    raise RuntimeError("Active set escaped frozen Proxy-48")

prompt_records=[]
for task_id, task in eval_challenges.items():
    user_prompt=build_agent_user_prompt(task)
    messages=[{"role":"system","content":SYSTEM_PROMPT},{"role":"user","content":user_prompt}]
    rendered, method=render_agent_chat(messages)
    prompt_records.append({
        "task_id":task_id, "user_prompt":user_prompt, "initial_messages":messages,
        "prompt":rendered, "prompt_sha256":hashlib.sha256(rendered.encode()).hexdigest(),
        "prompt_tokens":len(tokenizer.encode(rendered, add_special_tokens=False)),
        "n_train":len(task["train"]), "n_test":len(task["test"]), "render_method":method,
    })
prompt_df=pd.DataFrame([{k:v for k,v in r.items() if k not in {"prompt","initial_messages","user_prompt"}} for r in prompt_records])
prompt_df.to_csv(WORK_ROOT/"prompt_stats.csv", index=False)
PROMPT_RECORD_MAP={r["task_id"]:r for r in prompt_records if r["task_id"] in ACTIVE_TASK_SET}

chat_template=getattr(tokenizer,"chat_template","") or ""
template_sha=hashlib.sha256(chat_template.encode()).hexdigest()
if "<tool_call>" not in chat_template or "tool_response" not in chat_template:
    raise RuntimeError("Mounted Qwen chat template does not expose the required tool-call/tool-response contract.")

# Validate the *actual mounted tokenizer* on the multi-turn structure PIV will use.
_transport_probe_messages = [
    {"role":"system","content":SYSTEM_PROMPT},
    {"role":"user","content":"transport probe"},
    {"role":"assistant","content":"","reasoning_content":"PIV_REASONING_SENTINEL",
     "tool_calls":[{"type":"function","function":{"name":"python","arguments":{"code":"x=1\nx+1"}}}]},
    {"role":"tool","content":"2"},
]
_transport_rendered, _transport_method = render_agent_chat(_transport_probe_messages)
for _required in ("PIV_REASONING_SENTINEL", "<tool_call>", "<function=python>", "<parameter=code>", "<tool_response>", "2"):
    if _required not in _transport_rendered:
        raise RuntimeError(f"Qwen multi-turn tool-history rendering contract failed: missing {_required!r}")
if not _transport_rendered.rstrip().endswith("<think>"):
    raise RuntimeError("Qwen tool-history render did not reopen thinking for the next assistant turn.")
QWEN_TOOL_HISTORY_TRANSPORT_SHA256 = hashlib.sha256(_transport_rendered.encode("utf-8")).hexdigest()
with open(WORK_ROOT/"chat_template_fingerprint.txt","w") as f:
    f.write(f"sha256={template_sha}\ntool_schema_sha256={TOOL_SCHEMA_SHA256}\n")

profile_task_table=eval_feat[eval_feat["task_id"].isin(ACTIVE_TASK_IDS)].copy()
profile_task_table=profile_task_table.merge(proxy_source_feat,on="task_id",how="left")
profile_task_table["prompt_tokens_piv_initial"]=profile_task_table["task_id"].map(prompt_df.set_index("task_id")["prompt_tokens"])
profile_task_table["proxy_order"]=profile_task_table["task_id"].map(proxy48_order)
profile_task_table=profile_task_table.sort_values("proxy_order").reset_index(drop=True)
profile_task_table.to_csv(WORK_ROOT/"active_task_selection.csv",index=False)

SOURCE_ALIGNMENT={
    "experiment":EXPERIMENT_ID,
    "piv_source_repo":PIV_SOURCE_REPO,
    "piv_source_commit":PIV_SOURCE_COMMIT,
    "piv_source_files":PIV_SOURCE_FILES,
    "external_sources":PIV_EXTERNAL_SOURCES,
    "proxy48_ordered_sha256":PROXY48_ORDERED_SHA256,
    "source_file_sha256":SOURCE_FILE_SHA256,
    "tool_schema_sha256":TOOL_SCHEMA_SHA256,
    "chat_template_sha256":template_sha,
    "qwen_tool_history_transport_sha256":QWEN_TOOL_HISTORY_TRANSPORT_SHA256,
    "source_agentic_prompt_local_sha256":SOURCE_AGENTIC_PROMPT_LOCAL_SHA256,
    "source_system_prompt_local_sha256":SOURCE_SYSTEM_PROMPT_LOCAL_SHA256,
    "source_invariants":[
        "one persistent Python REPL per lineage",
        "puzzle preloaded without test outputs",
        "tool errors returned to model",
        "self-contained solution(grid) final artifact",
        "fresh independent all-train exact verification",
        "test grids produced by verified program execution",
    ],
    "local_adaptations":[
        "direct vLLM + Qwen native tool chat template; no OpenAI Responses server",
        "lightweight subprocess worker instead of Docker/IPyBox",
        "stricter grid/domain/time/file-system safety",
        "bounded 6-execution P0 tool budget and context guard",
        "source-style max_retries=2 reliability retries share the same total token/tool/turn budgets",
        "Proxy48 per-turn generation ceiling 4096; total generated-token cap remains 12288",
    ],
}
with open(WORK_ROOT/"SOURCE_ALIGNMENT.json","w") as f: json.dump(SOURCE_ALIGNMENT,f,indent=2)
print("Chat template/tool contract PASS | SHA:",template_sha)
print("ACTIVE TASKS:",len(ACTIVE_TASK_IDS),TASK_SET)
display(prompt_df[prompt_df.task_id.isin(ACTIVE_TASK_IDS)]["prompt_tokens"].describe().to_frame("initial_prompt_tokens").round(1))


In [ ]:
# =====================================================================================
# CELL 5 — PIV CONTEXT / COMPUTE SAFETY (preserve history; do not add P3 compaction in P0)
# =====================================================================================
active_prompt_tokens=prompt_df[prompt_df.task_id.isin(ACTIVE_TASK_IDS)]["prompt_tokens"].to_numpy()
max_initial_prompt=int(active_prompt_tokens.max())
CONTEXT_SAFETY_TOKENS=512
MAX_RENDERED_INPUT_TOKENS=MAX_MODEL_LEN-PER_TURN_MAX_TOKENS-CONTEXT_SAFETY_TOKENS
if max_initial_prompt > MAX_RENDERED_INPUT_TOKENS:
    raise RuntimeError(f"Initial PIV prompt exceeds context: {max_initial_prompt} > {MAX_RENDERED_INPUT_TOKENS}")
MAX_TOOL_OUTPUT_CHARS=4000
MAX_TOOL_RAW_ARTIFACT_CHARS=100000
MAX_CODE_CHARS=30000
print("PIV CONTEXT CONFIG")
print("  max_model_len           :",MAX_MODEL_LEN)
print("  max initial prompt      :",max_initial_prompt)
print("  per-turn max generation:",PER_TURN_MAX_TOKENS)
print("  total generated cap     :",TOTAL_GENERATED_TOKEN_CAP)
print("  max rendered input      :",MAX_RENDERED_INPUT_TOKENS)
print("  max tool output to model:",MAX_TOOL_OUTPUT_CHARS,"chars")
print("  history policy          : PRESERVE (P0); no P3 compact-state intervention")


In [ ]:
# =====================================================================================
# CELL 6 — QWEN TOOL XML PARSER + PERSISTENT LOCAL REPL + FRESH EXACT VERIFIER + SELF-TESTS
# =====================================================================================
import ast as py_ast
import re, subprocess, selectors, tempfile, uuid

DANGEROUS_IMPORT_ROOTS={"os","subprocess","pathlib","shutil","socket","requests","urllib","http","ftplib","paramiko","sys","builtins","importlib","inspect"}
DANGEROUS_CALL_NAMES={"open","exec","eval","compile","__import__","input","breakpoint","globals","locals","vars","getattr","setattr","delattr","exit","quit"}
DANGEROUS_DUNDER_ATTRS={"__subclasses__","__globals__","__builtins__","__code__","__closure__","__getattribute__"}

def canonical_grid(grid):
    if isinstance(grid,np.ndarray): grid=grid.tolist()
    if not is_rectangular_grid(grid): return None
    return tuple(tuple(int(x) for x in row) for row in grid)
def grids_equal(a,b):
    ca,cb=canonical_grid(a),canonical_grid(b)
    return ca is not None and cb is not None and ca==cb

def code_defines_solution(code):
    try: tree=py_ast.parse(code)
    except Exception: return False
    return any(isinstance(n,(py_ast.FunctionDef,py_ast.AsyncFunctionDef)) and n.name=="solution" for n in tree.body)

def extract_last_solution_code(text):
    blocks=re.findall(r"```(?:python|py)?\s*\n?(.*?)```", text or "", flags=re.I|re.S)
    for code in reversed(blocks):
        code=code.strip()
        if code_defines_solution(code): return code
    return None

def split_qwen_reasoning(text, action_pos=None):
    text=text or ""
    prefix=text if action_pos is None else text[:action_pos]
    if "</think>" in prefix:
        a,b=prefix.rsplit("</think>",1)
        return a.replace("<think>","",1).strip(), b.strip(), "think_close"
    # Generation prompt already opens <think>; without a close, pre-action text is reasoning.
    return prefix.replace("<think>","",1).strip(), "", "implicit_open_think"

def _action_region(text):
    """Prefer the model-action region after the final </think>; fall back to all text if no close exists."""
    text=text or ""
    if "</think>" in text:
        pos=text.rfind("</think>")+len("</think>")
        return text[pos:],pos
    return text,0

def parse_tool_calls(text):
    text=text or ""
    region,offset=_action_region(text)
    calls=[]
    pattern=r"<tool_call>\s*<function=([^>\s]+)>\s*(.*?)</function>\s*</tool_call>"
    for m in re.finditer(pattern,region,re.S):
        name=m.group(1).strip(); body=m.group(2); args={}
        for p in re.finditer(r"<parameter=([^>\s]+)>\s*(.*?)\s*</parameter>",body,re.S):
            args[p.group(1).strip()]=p.group(2)
        calls.append({"name":name,"arguments":args,"start":offset+m.start(),"end":offset+m.end(),"raw":m.group(0)})
    return calls

def parse_agent_turn(text):
    text=text or ""
    tcs=parse_tool_calls(text)
    if tcs:
        reasoning,content,rmode=split_qwen_reasoning(text,tcs[0]["start"])
        suffix=text[tcs[-1]["end"]:].strip()
        suffix=re.sub(r"(?:<\|im_end\|>|<\|endoftext\|>)\s*$","",suffix).strip()
        return {"kind":"tool","tool_calls":tcs,"tool_name":tcs[0]["name"],"tool_args":tcs[0]["arguments"],
                "reasoning_content":reasoning,"content":content,"reasoning_mode":rmode,
                "suffix_after_tool":suffix,"final_code":None}
    action_text,_=_action_region(text)
    if "<tool_call>" in action_text or "<function=" in action_text:
        reasoning,content,rmode=split_qwen_reasoning(text,None)
        return {"kind":"malformed_tool","tool_calls":[],"tool_name":None,"tool_args":{},
                "reasoning_content":reasoning,"content":content,"reasoning_mode":rmode,
                "suffix_after_tool":"","final_code":None}
    code=extract_last_solution_code(action_text)
    reasoning,content,rmode=split_qwen_reasoning(text,None)
    return {"kind":"final" if code else "text","tool_calls":[],"tool_name":None,"tool_args":{},
            "reasoning_content":reasoning,"content":content,"reasoning_mode":rmode,
            "suffix_after_tool":"","final_code":code}

def generation_finish_reason_ok(finish_reason):
    # Mirrors the source client's rule that a truncated response is not a valid turn result.
    return str(finish_reason).lower() == "stop"

def guard_python_code(code, *, final=False):
    if not isinstance(code,str) or not code.strip(): return False,"empty_code"
    if len(code)>MAX_CODE_CHARS: return False,"code_too_long"
    try: tree=py_ast.parse(code)
    except SyntaxError as e: return False,f"syntax_error:{e.msg}"
    for node in py_ast.walk(tree):
        if isinstance(node,py_ast.Name) and node.id=="__builtins__":
            return False,"blocked_builtins_escape"
        if isinstance(node,py_ast.Attribute) and node.attr in DANGEROUS_DUNDER_ATTRS:
            return False,f"blocked_dunder_attribute:{node.attr}"
        if isinstance(node,py_ast.Constant) and isinstance(node.value,str) and "/kaggle" in node.value.lower():
            return False,"blocked_host_path_literal"
        if isinstance(node,(py_ast.Import,py_ast.ImportFrom)):
            names=[a.name.split('.')[0] for a in node.names] if isinstance(node,py_ast.Import) else [(node.module or '').split('.')[0]]
            if any(x in DANGEROUS_IMPORT_ROOTS for x in names): return False,f"blocked_import:{names}"
        if isinstance(node,py_ast.Call):
            if isinstance(node.func,py_ast.Name) and node.func.id in DANGEROUS_CALL_NAMES:
                return False,f"blocked_call:{node.func.id}"
            if isinstance(node.func,py_ast.Attribute) and node.func.attr in DANGEROUS_DUNDER_ATTRS:
                return False,f"blocked_dunder_call:{node.func.attr}"
        if isinstance(node,py_ast.Raise):
            exc=node.exc
            if isinstance(exc,py_ast.Name) and exc.id=="SystemExit":
                return False,"blocked_system_exit"
            if isinstance(exc,py_ast.Call) and isinstance(exc.func,py_ast.Name) and exc.func.id=="SystemExit":
                return False,"blocked_system_exit"
        if isinstance(node,(py_ast.Assign,py_ast.AnnAssign,py_ast.AugAssign,py_ast.NamedExpr)):
            targets=[]
            if isinstance(node,py_ast.Assign): targets=node.targets
            else: targets=[node.target]
            def rooted_at_puzzle(t):
                while isinstance(t,(py_ast.Subscript,py_ast.Attribute)):
                    t=t.value
                return isinstance(t,py_ast.Name) and t.id=="puzzle"
            for t in targets:
                if rooted_at_puzzle(t): return False,"protected_puzzle_assignment"
        if isinstance(node,py_ast.Call) and isinstance(node.func,py_ast.Attribute):
            base=node.func.value
            while isinstance(base,(py_ast.Subscript,py_ast.Attribute)): base=base.value
            if isinstance(base,py_ast.Name) and base.id=="puzzle" and node.func.attr in {"update","clear","pop","popitem","setdefault","append","extend","insert","remove"}:
                return False,"protected_puzzle_mutation"
        if isinstance(node,py_ast.Expr) and isinstance(node.value,py_ast.Name) and node.value.id=="puzzle":
            return False,"blocked_whole_puzzle_display"
        if isinstance(node,py_ast.Call) and isinstance(node.func,py_ast.Name) and node.func.id=="print":
            if any(isinstance(a,py_ast.Name) and a.id=="puzzle" for a in node.args):
                return False,"blocked_whole_puzzle_print"
    if final and not code_defines_solution(code): return False,"missing_solution_function"
    return True,"ok"

def compact_tool_output(text,limit=MAX_TOOL_OUTPUT_CHARS):
    text=str(text or "")
    if len(text)<=limit: return text
    keep=max(1,(limit-120)//2)
    return text[:keep]+f"\n...[tool output truncated; original chars={len(text)}]...\n"+text[-keep:]

WORKER_SCRIPT = r"""
import sys,json,traceback,io,contextlib,copy,ast,os
def _piv_audit(event,args):
    try:
        if event=="open" and args:
            p=args[0]
            if isinstance(p,(str,bytes,os.PathLike)) and os.path.abspath(os.fsdecode(p)).startswith("/kaggle"):
                raise PermissionError("PIV sandbox blocks host /kaggle filesystem access")
        if event in {"os.listdir","os.scandir","os.chdir","os.remove","os.rename","os.rmdir","os.mkdir"} and args:
            p=args[0]
            if isinstance(p,(str,bytes,os.PathLike)) and os.path.abspath(os.fsdecode(p)).startswith("/kaggle"):
                raise PermissionError("PIV sandbox blocks host /kaggle filesystem access")
        if event=="subprocess.Popen" or event=="os.system" or event.startswith("socket."):
            raise PermissionError("PIV sandbox blocks process/network access")
    except PermissionError:
        raise
    except Exception:
        pass
sys.addaudithook(_piv_audit)
ns={"__name__":"__main__"}
original_puzzle=None
for line in sys.stdin:
    try:
        msg=json.loads(line); op=msg.get("op")
        if op=="init":
            original_puzzle=copy.deepcopy(msg["puzzle"])
            ns={"__name__":"__main__","puzzle":copy.deepcopy(original_puzzle)}
            out={"ok":True,"text":"REPL initialized."}
        elif op=="exec":
            # Restore the protected puzzle binding before every cell. Other REPL state persists.
            ns["puzzle"]=copy.deepcopy(original_puzzle)
            buf=io.StringIO()
            try:
                with contextlib.redirect_stdout(buf), contextlib.redirect_stderr(buf):
                    # Jupyter-like semantics: execute statements, then display repr(last expression).
                    tree=ast.parse(msg["code"],mode="exec")
                    if tree.body and isinstance(tree.body[-1],ast.Expr):
                        prefix=ast.Module(body=tree.body[:-1],type_ignores=[])
                        if prefix.body: exec(compile(prefix,"<piv-repl>","exec"),ns,ns)
                        value=eval(compile(ast.Expression(tree.body[-1].value),"<piv-repl>","eval"),ns,ns)
                        if value is not None: print(repr(value))
                    else:
                        exec(compile(tree,"<piv-repl>","exec"),ns,ns)
                out={"ok":True,"text":buf.getvalue() or "Code executed successfully. No output was returned."}
            except BaseException:
                out={"ok":False,"text":buf.getvalue()+traceback.format_exc(limit=8)}
        elif op=="close":
            print(json.dumps({"ok":True,"text":"closing"}),flush=True); break
        else: out={"ok":False,"text":"Unknown worker op"}
    except BaseException:
        out={"ok":False,"text":traceback.format_exc(limit=8)}
    print(json.dumps(out,ensure_ascii=False),flush=True)
"""

class PersistentPythonWorker:
    def __init__(self,puzzle,timeout_sec=10.0):
        self.puzzle=puzzle; self.timeout_sec=float(timeout_sec); self.proc=None; self.successful_cells=[]; self.restart_count=0
        self.start()
    def start(self):
        self.close(silent=True)
        self.proc=subprocess.Popen([sys.executable,"-u","-c",WORKER_SCRIPT],stdin=subprocess.PIPE,stdout=subprocess.PIPE,
                                   stderr=subprocess.DEVNULL,text=True,bufsize=1,cwd=tempfile.mkdtemp(prefix="piv_repl_"))
        r=self._request({"op":"init","puzzle":self.puzzle},timeout=max(2.0,self.timeout_sec))
        if not r.get("ok"): raise RuntimeError(f"REPL init failed: {r}")
        # Replay only previously successful bounded cells after a timeout/crash.
        if self.successful_cells:
            cells=list(self.successful_cells); self.successful_cells=[]
            for c in cells:
                rr=self.execute(c,record=True,replay=True)
                if not rr.get("ok"): break
    def _request(self,msg,timeout):
        if self.proc is None or self.proc.poll() is not None: raise RuntimeError("REPL process is not alive")
        self.proc.stdin.write(json.dumps(msg,ensure_ascii=False)+"\n"); self.proc.stdin.flush()
        sel=selectors.DefaultSelector(); sel.register(self.proc.stdout,selectors.EVENT_READ)
        events=sel.select(timeout); sel.close()
        if not events: raise TimeoutError(f"REPL tool exceeded {timeout}s")
        line=self.proc.stdout.readline()
        if not line: raise RuntimeError("REPL worker closed stdout")
        return json.loads(line)
    def execute(self,code,record=True,replay=False):
        ok,reason=guard_python_code(code,final=False)
        if not ok: return {"ok":False,"text":f"Tool code rejected by local safety guard: {reason}","guard_rejected":True,"restarted":False}
        try:
            r=self._request({"op":"exec","code":code},self.timeout_sec); r["restarted"]=False
            if r.get("ok") and record: self.successful_cells.append(code)
            return r
        except TimeoutError:
            self.restart_count+=1
            old=list(self.successful_cells)
            self.successful_cells=old
            try: self.start()
            except Exception: pass
            return {"ok":False,"text":f"Code execution terminated after {self.timeout_sec}s. Persistent worker was restarted; prior successful cells were replayed when possible.","timeout":True,"restarted":True}
        except Exception as e:
            self.restart_count+=1
            restarted=False
            try:
                self.start(); restarted=True
            except Exception:
                restarted=False
            return {"ok":False,"text":f"REPL infrastructure error: {type(e).__name__}: {e}. "
                                      f"Worker restart attempted; prior successful cells replayed when possible.",
                    "infrastructure_error":True,"restarted":restarted}
    def close(self,silent=False):
        p=getattr(self,"proc",None)
        if p is not None and p.poll() is None:
            try: p.terminate(); p.wait(timeout=1)
            except Exception:
                try: p.kill()
                except Exception: pass
        self.proc=None

VERIFIER_SCRIPT = r"""
import sys,json,copy,traceback,numbers,os
payload=json.loads(sys.stdin.read()); code=payload["code"]; task=payload["task"]
def _piv_audit(event,args):
    try:
        if event=="open" and args:
            p=args[0]
            if isinstance(p,(str,bytes,os.PathLike)) and os.path.abspath(os.fsdecode(p)).startswith("/kaggle"):
                raise PermissionError("PIV verifier blocks host /kaggle filesystem access")
        if event in {"os.listdir","os.scandir","os.chdir","os.remove","os.rename","os.rmdir","os.mkdir"} and args:
            p=args[0]
            if isinstance(p,(str,bytes,os.PathLike)) and os.path.abspath(os.fsdecode(p)).startswith("/kaggle"):
                raise PermissionError("PIV verifier blocks host /kaggle filesystem access")
        if event=="subprocess.Popen" or event=="os.system" or event.startswith("socket."):
            raise PermissionError("PIV verifier blocks process/network access")
    except PermissionError:
        raise
    except Exception:
        pass
sys.addaudithook(_piv_audit)
ns={"__name__":"__main__"}
def clean(g):
    try:
        import numpy as np
        if isinstance(g,np.ndarray): g=g.tolist()
    except Exception:
        pass
    # Source clean_grid accepts iterable rows. Normalize common iterable forms before LOCAL shape/domain checks.
    try:
        rows=list(g)
    except Exception:
        raise ValueError("invalid_grid_rows")
    if not (1<=len(rows)<=30):
        raise ValueError("invalid_grid_rows")
    out=[]
    for r in rows:
        try:
            if hasattr(r,"tolist"):
                r=r.tolist()
            rr0=list(r)
        except Exception:
            raise ValueError("invalid_grid_rows")
        out.append(rr0)
    widths=[len(r) for r in out]
    if not widths or min(widths)<1 or max(widths)>30 or len(set(widths))!=1:
        raise ValueError("invalid_grid_shape")
    clean_out=[]
    for r in out:
        rr=[]
        for x in r:
            if isinstance(x,bool) or not isinstance(x,numbers.Integral): raise ValueError("non_integer_cell")
            x=int(x)
            if x<0 or x>9: raise ValueError("cell_out_of_domain")
            rr.append(x)
        clean_out.append(rr)
    return clean_out

result={"ok":False,"all_train_exact":False,"train":[],"test_outputs":[],
        "test_execution_ok":False,"any_test_output_ok":False,"test_errors":[],
        "train_error":None,"test_error":None}
try:
    exec(code,ns,ns); fn=ns.get("solution")
    if not callable(fn): raise ValueError("solution_not_callable")
except Exception:
    result["train_error"]=traceback.format_exc(limit=10)
    print(json.dumps(result)); raise SystemExit(0)

# SOURCE-FAITHFUL ordering: establish the train gate first.
for i,ex in enumerate(task["train"]):
    try:
        pred=clean(fn(copy.deepcopy(ex["input"])))
        exact=(pred==ex["output"])
        result["train"].append({"index":i,"exact":exact,"pred":pred if not exact else None,"error":None})
    except Exception:
        result["train"].append({"index":i,"exact":False,"pred":None,"error":traceback.format_exc(limit=8)})
result["all_train_exact"]=bool(result["train"]) and all(x["exact"] for x in result["train"])

# Test execution is a separate per-output state, matching source produce_output_grid.py.
# One failing test input must not erase successful outputs for other test inputs.
if result["all_train_exact"]:
    tests=[]; test_errors=[]
    for j,ex in enumerate(task["test"]):
        try:
            tests.append(clean(fn(copy.deepcopy(ex["input"]))))
            test_errors.append(None)
        except Exception:
            tests.append(None)
            test_errors.append(traceback.format_exc(limit=10))
    result["test_outputs"]=tests
    result["test_errors"]=test_errors
    result["test_execution_ok"]=all(x is not None for x in tests)
    result["any_test_output_ok"]=any(x is not None for x in tests)
    result["ok"]=result["any_test_output_ok"]
    if not result["test_execution_ok"]:
        result["test_error"]="\n".join(e for e in test_errors if e)

print(json.dumps(result))
"""

def task_without_test_solutions(task):
    return {"train":copy.deepcopy(task["train"]),"test":[{"input":copy.deepcopy(x["input"])} for x in task["test"]]}

def fresh_verify_solution(code,task,timeout_sec=VERIFIER_TIMEOUT_SEC):
    ok,reason=guard_python_code(code,final=True)
    if not ok: return {"ok":False,"all_train_exact":False,"train":[],"test_outputs":[],"error":reason,"verifier_status":"guard_rejected"}
    payload={"code":code,"task":task_without_test_solutions(task)}
    try:
        cp=subprocess.run([sys.executable,"-u","-c",VERIFIER_SCRIPT],input=json.dumps(payload),text=True,capture_output=True,
                          timeout=float(timeout_sec),cwd=tempfile.mkdtemp(prefix="piv_verify_"))
        if cp.returncode!=0: return {"ok":False,"all_train_exact":False,"train":[],"test_outputs":[],"error":cp.stderr[-4000:],"verifier_status":"process_error"}
        line=cp.stdout.strip().splitlines()[-1] if cp.stdout.strip() else ""
        r=json.loads(line)
        if r.get("all_train_exact") and r.get("test_execution_ok"):
            r["verifier_status"]="train_exact_test_ok"
        elif r.get("all_train_exact") and r.get("any_test_output_ok"):
            r["verifier_status"]="train_exact_test_partial"
        elif r.get("all_train_exact"):
            r["verifier_status"]="train_exact_test_failed"
        else:
            r["verifier_status"]="train_rejected"
        return r
    except subprocess.TimeoutExpired:
        return {"ok":False,"all_train_exact":False,"train":[],"test_outputs":[],"error":"fresh_verifier_timeout","verifier_status":"timeout"}
    except Exception as e:
        return {"ok":False,"all_train_exact":False,"train":[],"test_outputs":[],"error":f"{type(e).__name__}: {e}","verifier_status":"error"}

# ---------------------------- PRELOAD SELF-TESTS ----------------------------
self_tests={}
t=parse_agent_turn("reasoning\n</think>\n<tool_call>\n<function=python>\n<parameter=code>\npuzzle.keys()\n</parameter>\n</function>\n</tool_call>")
assert t["kind"]=="tool" and len(t["tool_calls"])==1 and t["tool_name"]=="python" and "puzzle.keys" in t["tool_args"]["code"]
t_multi=parse_agent_turn("r\n</think>\n<tool_call>\n<function=python>\n<parameter=code>\nx=1\n</parameter>\n</function>\n</tool_call>\n<tool_call>\n<function=python>\n<parameter=code>\nx+1\n</parameter>\n</function>\n</tool_call>")
assert t_multi["kind"]=="tool" and len(t_multi["tool_calls"])==2
t_bad=parse_agent_turn("<tool_call><function=python><parameter=code>x=1</function></tool_call>")
assert t_bad["kind"]=="malformed_tool" or (t_bad["kind"]=="tool" and "code" not in t_bad["tool_args"])
t_rehearse=parse_agent_turn(
    "<think>maybe <tool_call><function=python><parameter=code>print('wrong')</parameter></function></tool_call></think>"
    "<tool_call><function=python><parameter=code>print('right')</parameter></function></tool_call>")
assert t_rehearse["kind"]=="tool" and len(t_rehearse["tool_calls"])==1 and "right" in t_rehearse["tool_calls"][0]["arguments"]["code"]
assert generation_finish_reason_ok("stop") and not generation_finish_reason_ok("length")
self_tests["qwen_tool_xml_parser_and_finish_guard"]=True
f=parse_agent_turn("done\n</think>\n```python\ndef solution(grid):\n    return grid\n```")
assert f["kind"]=="final" and code_defines_solution(f["final_code"])
self_tests["final_solution_extractor"]=True
assert guard_python_code("puzzle.keys()")[0]
assert not guard_python_code("puzzle = {}",final=False)[0]
assert not guard_python_code("puzzle[\"train\"] = []",final=False)[0]
assert not guard_python_code("import os\nprint(os.listdir('.'))",final=False)[0]
assert not guard_python_code('print("/kaggle/input/secret.json")',final=False)[0]
assert guard_python_code("class A:\n    def __init__(self): self.x=1\nclass B(A):\n    def __init__(self): super().__init__()",final=False)[0]
assert not guard_python_code("object.__subclasses__()",final=False)[0]
assert not guard_python_code("raise SystemExit('bye')",final=False)[0]
self_tests["tool_safety_guard"]=True
_fake={"train":[{"input":[[1]],"output":[[1]]}],"test":[{"input":[[2]]}]}
w=PersistentPythonWorker(task_without_test_solutions(_fake),timeout_sec=5.0)
r0=w.execute("puzzle.keys()"); r1=w.execute("x=41"); r2=w.execute("x+1")
assert r0["ok"] and "dict_keys" in r0["text"] and r1["ok"] and r2["ok"] and "42" in r2["text"]
ra=w.execute("g=puzzle['train'][0]['input']; g[0][0]=9")
rb=w.execute("puzzle['train'][0]['input'][0][0]")
assert rb["ok"] and "1" in rb["text"]
rc=w.execute("raise SystemExit('bye')")
rd=w.execute("1+1")
assert (not rc["ok"]) and rd["ok"] and "2" in rd["text"]
w.close()
self_tests["persistent_repl_state_and_expression_display"]=True
v=fresh_verify_solution("def solution(grid):\n    return [row[:] for row in grid]",_fake,timeout_sec=5)
assert v["all_train_exact"] and v["test_outputs"]==[[[2]]]
self_tests["fresh_exact_verifier"]=True
v2=fresh_verify_solution("def solution(grid):\n    return [[9]]",_fake,timeout_sec=5)
assert not v2["all_train_exact"] and v2["test_outputs"]==[]
self_tests["train_fail_blocks_test_execution"]=True
v3=fresh_verify_solution(
    "def solution(grid):\n    if grid == [[2]]: raise RuntimeError('test-only failure')\n    return tuple(tuple(x for x in row) for row in grid)",
    _fake,timeout_sec=5)
assert v3["all_train_exact"] and not v3["test_execution_ok"] and v3["test_outputs"]==[None]
self_tests["train_exact_survives_test_execution_failure"]=True
_fake_multi={"train":[{"input":[[1]],"output":[[1]]}],"test":[{"input":[[2]]},{"input":[[3]]}]}
v4=fresh_verify_solution(
    "def solution(grid):\n    if grid == [[2]]: raise RuntimeError('one test fails')\n    return tuple(tuple(x for x in row) for row in grid)",
    _fake_multi,timeout_sec=5)
assert v4["all_train_exact"] and v4["test_outputs"][0] is None and v4["test_outputs"][1]==[[3]]
self_tests["partial_multiquery_test_outputs_preserved"]=True
with open(WORK_ROOT/"preload_self_tests.json","w") as f: json.dump(self_tests,f,indent=2)
print("PIV preload self-tests PASS:",self_tests)


In [ ]:
# =====================================================================================
# CELL 7 — CPU-SAFE PREFLIGHT OR REAL vLLM/TP=4 MODEL LOAD (Q1-FROZEN ENGINE)
# Dependency versions were already pinned in Cell 1 for PREFLIGHT/production parity.
# =====================================================================================
gpu_info = []
notebook_start = NOTEBOOK_WALL_START
model_load_sec = 0.0
warmup_sec = 0.0
arc_llm = None

RUNTIME_VLLM_VERSION = pkg_version("vllm")
RUNTIME_TORCH_VERSION = pkg_version("torch")
RUNTIME_TRANSFORMERS_VERSION = pkg_version("transformers")
RUNTIME_TOKENIZERS_VERSION = pkg_version("tokenizers")
RUNTIME_HF_HUB_VERSION = pkg_version("huggingface-hub")
RUNTIME_CUDA_VERSION = None

deadline = notebook_start + TOTAL_RUNTIME_SEC - SAFETY_RESERVE_SEC

# Fingerprint the mounted checkpoint without hashing the huge shard bytes.
with open(index_path, "r", encoding="utf-8") as f:
    _model_index = json.load(f)
_model_shards = sorted(set((_model_index.get("weight_map") or {}).values()))
_model_shard_layout = []
for _name in _model_shards:
    _p = MODEL_PATH / _name
    _model_shard_layout.append({
        "name": _name,
        "bytes": int(_p.stat().st_size) if _p.exists() else None,
    })

MODEL_ARTIFACT_FINGERPRINT_PAYLOAD = {
    "model_path": str(MODEL_PATH),
    "config_sha256": small_file_sha256(config_path),
    "safetensors_index_sha256": small_file_sha256(index_path),
    "shard_layout": _model_shard_layout,
    "chat_template_sha256": template_sha,
    "tokenizer_class": tokenizer.__class__.__name__,
    "tokenizer_vocab_size": len(tokenizer),
}
MODEL_ARTIFACT_FINGERPRINT_SHA256 = stable_json_sha256(MODEL_ARTIFACT_FINGERPRINT_PAYLOAD)

ENGINE_CONFIG = {
    "model": str(MODEL_PATH),
    "tensor_parallel_size": 4,
    "distributed_executor_backend": "mp",
    "dtype": "auto",
    "load_format": "safetensors",
    "trust_remote_code": False,
    "language_model_only": True,
    "max_model_len": MAX_MODEL_LEN,
    "max_num_seqs": MAX_NUM_SEQS,
    "max_num_batched_tokens": MAX_NUM_BATCHED_TOKENS,
    "gpu_memory_utilization": GPU_MEMORY_UTILIZATION,
    "enforce_eager": True,
    "disable_custom_all_reduce": True,
    "enable_prefix_caching": False,
    "seed": SEED,
}
ENGINE_CONFIG_SHA256 = stable_json_sha256(ENGINE_CONFIG)

if not USE_MODEL:
    # True no-allocation path: package parity is real, but do not import vLLM/CUDA runtime.
    SamplingParams = None
    print("=" * 116)
    print("PREFLIGHT CPU-SAFE / DEPENDENCY-PARITY MODE")
    print("=" * 116)
    print("vLLM package metadata :", RUNTIME_VLLM_VERSION)
    print("Transformers          :", RUNTIME_TRANSFORMERS_VERSION)
    print("Tokenizers            :", RUNTIME_TOKENIZERS_VERSION)
    print("HF Hub                :", RUNTIME_HF_HUB_VERSION)
    print("Skipping vLLM import, CUDA checks, SamplingParams API check and 27B allocation.")
    print("GPU/runtime-specific paths remain covered by SMOKE_GPU.")
else:
    import torch
    import vllm
    from vllm import LLM, SamplingParams

    RUNTIME_VLLM_VERSION = vllm.__version__
    RUNTIME_TORCH_VERSION = torch.__version__
    RUNTIME_CUDA_VERSION = torch.version.cuda

    print("=" * 116)
    print("REAL vLLM / CUDA PREFLIGHT")
    print("=" * 116)
    print("PyTorch :", RUNTIME_TORCH_VERSION)
    print("CUDA    :", RUNTIME_CUDA_VERSION)
    print("vLLM    :", RUNTIME_VLLM_VERSION)
    print("Transformers:", pkg_version("transformers"))

    if RUNTIME_VLLM_VERSION != EXPECTED_RUNTIME_PACKAGES["vllm"]:
        raise RuntimeError("vLLM runtime drift after import.")
    if pkg_version("transformers") != EXPECTED_RUNTIME_PACKAGES["transformers"]:
        raise RuntimeError("Transformers runtime drift after vLLM setup.")

    # Fail fast on the exact SamplingParams fields used later, BEFORE allocating 27B.
    _sampling_api_probe = SamplingParams(
        temperature=1.0,
        top_p=0.95,
        top_k=20,
        min_p=0.0,
        presence_penalty=0.0,
        repetition_penalty=1.0,
        max_tokens=16,
        ignore_eos=False,
        seed=SEED,
        skip_special_tokens=False,
        spaces_between_special_tokens=False,
    )
    print("SamplingParams API self-test: PASS")

    print("\nLOADING QWEN3.8-27B-FP8 ON 4×L4")
    print("GPUs visible:", torch.cuda.device_count())

    if not torch.cuda.is_available():
        raise RuntimeError("CUDA unavailable. Select Kaggle 4×L4 accelerator.")
    if torch.cuda.device_count() != 4:
        raise RuntimeError(f"Expected 4 GPUs; PyTorch sees {torch.cuda.device_count()}.")

    for i in range(4):
        cc = torch.cuda.get_device_capability(i)
        mem = torch.cuda.get_device_properties(i).total_memory / 1024**3
        name = torch.cuda.get_device_name(i)
        gpu_info.append({"gpu": i, "name": name, "memory_gib": mem, "cc": f"{cc[0]}.{cc[1]}"})
        print(f"GPU {i}: {name} | {mem:.2f} GiB | CC {cc[0]}.{cc[1]}")

    load_start = time.perf_counter()
    arc_llm = LLM(**ENGINE_CONFIG)
    model_load_sec = time.perf_counter() - load_start
    print(f"MODEL LOAD COMPLETE: {model_load_sec:.1f} sec")

    # Real generation smoke request catches engine/scheduler/model problems immediately.
    _active_prompts = [r for r in prompt_records if r["task_id"] in ACTIVE_TASK_SET]
    _med = float(np.median([r["prompt_tokens"] for r in _active_prompts]))
    probe_record = min(_active_prompts, key=lambda r: abs(r["prompt_tokens"] - _med))
    warm_params = SamplingParams(
        temperature=0.0,
        max_tokens=32,
        ignore_eos=False,
        skip_special_tokens=False,
    )
    t0 = time.perf_counter()
    warm = arc_llm.generate([probe_record["prompt"]], [warm_params], use_tqdm=False)
    warmup_sec = time.perf_counter() - t0
    assert len(warm) == 1 and len(warm[0].outputs) >= 1
    print(f"REAL MODEL WARMUP (per-request SamplingParams sequence): PASS ({warmup_sec:.2f} sec)")

RUNTIME_STACK = {
    "vllm": pkg_version("vllm"),
    "torch": pkg_version("torch"),
    "transformers": pkg_version("transformers"),
    "tokenizers": pkg_version("tokenizers"),
    "huggingface-hub": pkg_version("huggingface-hub"),
    "cuda": RUNTIME_CUDA_VERSION,
}
RUNTIME_STACK_SHA256 = stable_json_sha256(RUNTIME_STACK)

print("Model artifact fingerprint:", MODEL_ARTIFACT_FINGERPRINT_SHA256)
print("Engine config fingerprint :", ENGINE_CONFIG_SHA256)
print("Runtime stack fingerprint :", RUNTIME_STACK_SHA256)
print("Safety reserve:", SAFETY_RESERVE_SEC / 60, "minutes")


In [ ]:
# =====================================================================================
# CELL 8 — PIV-Q27-P0 BOUNDED STATEFUL AGENT LOOP + CONTRACT-SAFE CHECKPOINTS
# =====================================================================================
RAW_GENERATIONS_PATH=WORK_ROOT/"raw_generations.jsonl"
TRACE_DIR=WORK_ROOT/"traces"; TRACE_DIR.mkdir(exist_ok=True)

PIV_SAMPLING={
    "temperature":1.0,"top_p":0.95,"top_k":20,"min_p":0.0,
    "presence_penalty":0.0,"repetition_penalty":1.0,"seed":SEED+3000,
    "per_turn_max_tokens":PER_TURN_MAX_TOKENS,
}
QWEN_OFFICIAL_SAMPLING_REFERENCE={"thinking":{k:PIV_SAMPLING[k] for k in ["temperature","top_p","top_k","min_p","presence_penalty","repetition_penalty"]}}
SAMPLING_CONFIG_SHA256=stable_json_sha256(PIV_SAMPLING)
AGENT_BUDGET={
    "agent_batch_size":AGENT_BATCH_SIZE,"max_tool_calls":MAX_TOOL_CALLS,"max_model_turns":MAX_MODEL_TURNS,
    "max_retries":MAX_RETRIES,
    "per_turn_max_tokens":PER_TURN_MAX_TOKENS,"total_generated_token_cap":TOTAL_GENERATED_TOKEN_CAP,
    "tool_timeout_sec":TOOL_TIMEOUT_SEC,"verifier_timeout_sec":VERIFIER_TIMEOUT_SEC,
    "max_tool_output_chars":MAX_TOOL_OUTPUT_CHARS,"history_policy":"preserve_thinking_and_tool_history",
}
AGENT_BUDGET_SHA256=stable_json_sha256(AGENT_BUDGET)

ENGINE_RUNTIME_CONTRACT={
    "model_artifact_fingerprint_sha256":MODEL_ARTIFACT_FINGERPRINT_SHA256,
    "runtime_stack_sha256":RUNTIME_STACK_SHA256,
    "engine_config_sha256":ENGINE_CONFIG_SHA256,
    "chat_template_sha256":template_sha,
    "tool_schema_sha256":TOOL_SCHEMA_SHA256,
}

def generation_contract_payload(task_id):
    pr=PROMPT_RECORD_MAP[task_id]
    return {
        "generation_contract_version":GENERATION_CONTRACT_VERSION,"parser_contract_version":PARSER_CONTRACT_VERSION,
        "prompt_contract_version":PROMPT_CONTRACT_VERSION,"experiment_id":EXPERIMENT_ID,"run_profile":RUN_PROFILE,
        "task_set":TASK_SET,"task_id":task_id,"n_train":pr["n_train"],"n_test":pr["n_test"],
        "initial_prompt_sha256":pr["prompt_sha256"],"piv_source_commit":PIV_SOURCE_COMMIT,
        "sampling_config_sha256":SAMPLING_CONFIG_SHA256,"agent_budget_sha256":AGENT_BUDGET_SHA256,
        **ENGINE_RUNTIME_CONTRACT,
    }
def current_contract_sha(task_id): return stable_json_sha256(generation_contract_payload(task_id))

def append_record(rec):
    with open(RAW_GENERATIONS_PATH,"a",encoding="utf-8") as f:
        f.write(json.dumps(rec,ensure_ascii=False,default=str)+"\n")

def read_existing_records():
    if not RAW_GENERATIONS_PATH.exists(): return []
    out=[]
    for i,line in enumerate(RAW_GENERATIONS_PATH.read_text(encoding="utf-8").splitlines()):
        if not line.strip(): continue
        try:
            r=json.loads(line); r["_checkpoint_order"]=i; out.append(r)
        except Exception: pass
    return out

def record_compatible(r):
    tid=r.get("task_id")
    if tid not in ACTIVE_TASK_SET: return False,"foreign_task"
    if r.get("generation_contract_sha256") != current_contract_sha(tid): return False,"contract_mismatch"
    return True,"ok"

def audit_checkpoint_records(records):
    compatible=[]; stale=[]; foreign=[]
    for r in records:
        ok,reason=record_compatible(r)
        if ok: compatible.append(r)
        elif reason=="foreign_task": foreign.append({"task_id":r.get("task_id"),"record_id":r.get("record_id")})
        else: stale.append({"task_id":r.get("task_id"),"record_id":r.get("record_id"),"reason":reason})
    return compatible,{"raw_rows":len(records),"compatible_rows":len(compatible),"stale_rows":len(stale),"foreign_rows":len(foreign),
                       "stale_preview":stale[:20],"foreign_preview":foreign[:20]}

def terminal_completed_task_ids(records):
    out=set()
    for r in records:
        ok,_=record_compatible(r)
        if ok and r.get("status")=="complete": out.add(r["task_id"])
    return out

def save_event(task_id,event):
    p=TRACE_DIR/f"{task_id}.jsonl"
    with open(p,"a",encoding="utf-8") as f: f.write(json.dumps(event,ensure_ascii=False,default=str)+"\n")

def assistant_tool_message(parsed):
    calls=[]
    for tc in parsed.get("tool_calls",[]):
        calls.append({"type":"function","function":{"name":tc["name"],"arguments":tc["arguments"]}})
    return {
        "role":"assistant","content":parsed.get("content","") or "",
        "reasoning_content":parsed.get("reasoning_content","") or "",
        "tool_calls":calls,
    }

def complete_record(state, task, verifier, candidate_status, stop_reason, error=""):
    test_outputs=verifier.get("test_outputs",[]) if verifier else []
    code=state.get("final_code")
    rec={
        "run_id":RUN_ID,"record_id":str(uuid.uuid4()),"task_id":state["task_id"],"branch":"piv_p0","status":"complete",
        "candidate_status":candidate_status,"stop_reason":stop_reason,"error":error,
        "n_train":len(task["train"]),"n_test":len(task["test"]),"seed":PIV_SAMPLING["seed"],
        "sampling_seeds":state.get("sampling_seeds",[]),"source_retries":state.get("source_retries",0),
        "step_retries_total":state.get("step_retries_total",0),"retry_reasons":state.get("retry_reasons",[]),
        "budget_denials":state.get("budget_denials",0),
        "model_turns":state["model_turns"],"tool_calls":state["tool_calls"],"tool_errors":state["tool_errors"],
        "tool_timeouts":state["tool_timeouts"],"worker_restarts":state["worker_restarts"],
        "generated_tokens":state["generated_tokens"],"max_rendered_prompt_tokens":state["max_prompt_tokens"],
        "finish_reasons":state["finish_reasons"],"final_code":code,"final_code_sha256":hashlib.sha256((code or "").encode()).hexdigest() if code else None,
        "all_train_exact":bool(verifier and verifier.get("all_train_exact")),
        "test_execution_ok":bool(verifier and verifier.get("test_execution_ok")),
        "any_test_output_ok":bool(verifier and verifier.get("any_test_output_ok")),
        "test_errors":verifier.get("test_errors",[]) if verifier else [],
        "verifier_status":verifier.get("verifier_status") if verifier else None,
        "verifier_error":(verifier.get("train_error") or verifier.get("test_error") or verifier.get("error")) if verifier else None,
        "test_outputs":test_outputs,
        "initial_prompt_sha256":PROMPT_RECORD_MAP[state["task_id"]]["prompt_sha256"],
        "generation_contract_sha256":current_contract_sha(state["task_id"]),"sampling_config_sha256":SAMPLING_CONFIG_SHA256,
        "agent_budget_sha256":AGENT_BUDGET_SHA256,"piv_source_commit":PIV_SOURCE_COMMIT,**ENGINE_RUNTIME_CONTRACT,
    }
    return rec

def run_scripted_preflight_agent_loop_test():
    """CPU-only scripted tool->feedback->final-code path using the same parser/worker/verifier pieces."""
    fake={"train":[{"input":[[1,2],[3,4]],"output":[[1,2],[3,4]]}],
          "test":[{"input":[[5,6],[7,8]]}]}
    worker=PersistentPythonWorker(task_without_test_solutions(fake),timeout_sec=5.0)
    try:
        raw1="inspect\n</think>\n<tool_call><function=python><parameter=code>puzzle.keys()</parameter></function></tool_call>"
        p1=parse_agent_turn(raw1)
        assert p1["kind"]=="tool" and len(p1["tool_calls"])==1
        rr=worker.execute(p1["tool_calls"][0]["arguments"]["code"])
        assert rr["ok"] and "dict_keys" in rr["text"]
        raw2="done\n</think>\n```python\ndef solution(grid):\n    return [row[:] for row in grid]\n```"
        p2=parse_agent_turn(raw2)
        assert p2["kind"]=="final"
        vr=fresh_verify_solution(p2["final_code"],fake,timeout_sec=5)
        assert vr["all_train_exact"] and vr["test_outputs"]==[[[5,6],[7,8]]]
        payload={"ok":True,"tool_feedback":rr["text"],"verifier_status":vr["verifier_status"]}
    finally:
        worker.close()
    with open(WORK_ROOT/"scripted_agent_loop_selftest.json","w") as f:
        json.dump(payload,f,indent=2)
    print("Scripted CPU agent-loop self-test PASS:",payload)

def run_mock_preflight():
    run_scripted_preflight_agent_loop_test()
    # Exercise downstream checkpoint/scoring paths without pretending to measure Qwen accuracy.
    for tid in ACTIVE_TASK_IDS:
        state={"task_id":tid,"model_turns":0,"tool_calls":0,"tool_errors":0,"tool_timeouts":0,"worker_restarts":0,
               "generated_tokens":0,"max_prompt_tokens":PROMPT_RECORD_MAP[tid]["prompt_tokens"],
               "finish_reasons":["mock_preflight"],"final_code":None,"sampling_seeds":[],
               "source_retries":0,"step_retries_total":0,"retry_reasons":[],"budget_denials":0}
        rec=complete_record(state,eval_challenges[tid],None,"mock_rejected","mock_preflight")
        append_record(rec)
    print("PREFLIGHT mock records written:",len(ACTIVE_TASK_IDS))

def run_real_piv():
    existing=read_existing_records(); completed=terminal_completed_task_ids(existing)
    pending=[t for t in ACTIVE_TASK_IDS if t not in completed]
    print("Resume: completed",len(completed),"pending",len(pending))

    for batch_no,start_idx in enumerate(range(0,len(pending),AGENT_BATCH_SIZE),1):
        tids=pending[start_idx:start_idx+AGENT_BATCH_SIZE]
        if time.perf_counter()>=deadline:
            print("Safety deadline reached before batch",batch_no); break
        workers={}; states={}

        def start_fresh_attempt(tid, st, reason=None):
            # PiVOTools AbstractTurn retries invalid/incomplete agentic calls from the initial prompt.
            # LOCAL fairness: retries share the same total token/tool/turn budgets; they do not create extra candidates.
            if reason is not None:
                if st["source_retries"]>=MAX_RETRIES:
                    return False
                if st["generated_tokens"]>=TOTAL_GENERATED_TOKEN_CAP or st["model_turns"]>=MAX_MODEL_TURNS:
                    return False
                st["source_retries"]+=1
                st["sampling_retry_nonce"]+=1
                st["retry_reasons"].append(str(reason))
            old=workers.get(tid)
            if old is not None:
                st["worker_restart_offset"] += int(getattr(old,"restart_count",0))
                st["worker_restarts"] = st["worker_restart_offset"]
                try: old.close()
                except Exception: pass
            workers[tid]=PersistentPythonWorker(task_without_test_solutions(eval_challenges[tid]),TOOL_TIMEOUT_SEC)
            st["messages"]=copy.deepcopy(PROMPT_RECORD_MAP[tid]["initial_messages"])
            st["attempt_turns"]=0
            st["step_retry_count"]=0
            st["budget_nudged"]=False
            st["final_code"]=None
            return True

        try:
            for tid in tids:
                states[tid]={
                    "task_id":tid,"messages":[],"model_turns":0,"attempt_turns":0,
                    "tool_calls":0,"tool_errors":0,"tool_timeouts":0,"worker_restarts":0,"worker_restart_offset":0,
                    "generated_tokens":0,"max_prompt_tokens":0,"finish_reasons":[],"done":False,"final_code":None,
                    "source_retries":0,"step_retry_count":0,"step_retries_total":0,"sampling_retry_nonce":0,
                    "sampling_seeds":[],"retry_reasons":[],"budget_nudged":False,"budget_denials":0,
                }
                start_fresh_attempt(tid,states[tid])

            while True:
                active=[t for t in tids if not states[t]["done"]]
                if not active: break

                if time.perf_counter()>=deadline:
                    for tid in active:
                        st=states[tid]; st["done"]=True
                        rec=complete_record(st,eval_challenges[tid],None,"failed","global_deadline","safety reserve reached")
                        rec["status"]="failed"; append_record(rec)
                    break

                render=[]; render_tids=[]; max_allowed=[]; request_seeds=[]
                for tid in active:
                    st=states[tid]
                    if st["model_turns"]>=MAX_MODEL_TURNS or st["generated_tokens"]>=TOTAL_GENERATED_TOKEN_CAP:
                        st["done"]=True
                        rec=complete_record(st,eval_challenges[tid],None,"rejected","agent_budget_exhausted")
                        append_record(rec); continue

                    prompt,method=render_agent_chat(st["messages"])
                    pt=len(tokenizer.encode(prompt,add_special_tokens=False))
                    st["max_prompt_tokens"]=max(st["max_prompt_tokens"],pt)
                    safe=MAX_MODEL_LEN-pt-CONTEXT_SAFETY_TOKENS
                    remaining=TOTAL_GENERATED_TOKEN_CAP-st["generated_tokens"]
                    cap=min(PER_TURN_MAX_TOKENS,remaining,safe)
                    if cap<32:
                        st["done"]=True
                        rec=complete_record(st,eval_challenges[tid],None,"rejected","context_or_token_budget_exhausted")
                        append_record(rec); continue

                    req_seed=int(PIV_SAMPLING["seed"]+st["sampling_retry_nonce"])
                    render.append(prompt); render_tids.append(tid); max_allowed.append(int(cap)); request_seeds.append(req_seed)
                    st["sampling_seeds"].append(req_seed)

                if not render_tids: continue

                params=[]
                for _cap,_seed in zip(max_allowed,request_seeds):
                    params.append(SamplingParams(
                        temperature=PIV_SAMPLING["temperature"],top_p=PIV_SAMPLING["top_p"],top_k=PIV_SAMPLING["top_k"],
                        min_p=PIV_SAMPLING["min_p"],presence_penalty=PIV_SAMPLING["presence_penalty"],
                        repetition_penalty=PIV_SAMPLING["repetition_penalty"],max_tokens=int(_cap),ignore_eos=False,
                        seed=int(_seed),skip_special_tokens=False,spaces_between_special_tokens=False))

                t0=time.perf_counter(); outs=arc_llm.generate(render,params,use_tqdm=False); wave_sec=time.perf_counter()-t0

                for tid,req in zip(render_tids,outs):
                    st=states[tid]; st["model_turns"]+=1; st["attempt_turns"]+=1

                    if not getattr(req,"outputs",None):
                        event={"run_id":RUN_ID,"task_id":tid,"turn":st["model_turns"],"attempt_turn":st["attempt_turns"],
                               "parsed_kind":"no_output","wave_sec":wave_sec}
                        if start_fresh_attempt(tid,st,"no_output"):
                            event["source_retry"]=True; save_event(tid,event); continue
                        st["done"]=True
                        rec=complete_record(st,eval_challenges[tid],None,"failed","no_output","vLLM returned no output")
                        rec["status"]="failed"; append_record(rec); save_event(tid,event); continue

                    o=req.outputs[0]; text=o.text or ""; nt=len(getattr(o,"token_ids",[]) or [])
                    st["generated_tokens"]+=nt
                    fr=str(getattr(o,"finish_reason",None)); st["finish_reasons"].append(fr)
                    parsed=parse_agent_turn(text)
                    event={"run_id":RUN_ID,"task_id":tid,"turn":st["model_turns"],"attempt_turn":st["attempt_turns"],
                           "source_retry_index":st["source_retries"],"prompt_tokens":st["max_prompt_tokens"],
                           "generated_tokens":nt,"finish_reason":fr,"raw_text":text,"parsed_kind":parsed["kind"],
                           "wave_sec":wave_sec,"request_seed":st["sampling_seeds"][-1]}

                    # Source client rejects incomplete responses; AbstractTurn then retries the complete agentic call.
                    if not generation_finish_reason_ok(fr):
                        if start_fresh_attempt(tid,st,f"invalid_finish_reason:{fr}"):
                            event["source_retry"]=True; save_event(tid,event); continue
                        st["done"]=True
                        rec=complete_record(st,eval_challenges[tid],None,"rejected","invalid_finish_reason",
                                            f"finish_reason={fr}; retries exhausted or budget unavailable")
                        append_record(rec); save_event(tid,event); continue

                    if parsed["kind"]=="malformed_tool":
                        # Source ChatCompletions client retries malformed tool-call formatting before abandoning the call.
                        if st["step_retry_count"]<MAX_RETRIES and st["generated_tokens"]<TOTAL_GENERATED_TOKEN_CAP and st["model_turns"]<MAX_MODEL_TURNS:
                            st["step_retry_count"]+=1; st["step_retries_total"]+=1; st["sampling_retry_nonce"]+=1
                            event["step_retry"]=True; save_event(tid,event); continue
                        if start_fresh_attempt(tid,st,"malformed_tool_xml"):
                            event["source_retry"]=True; save_event(tid,event); continue
                        st["tool_errors"]+=1; st["done"]=True
                        rec=complete_record(st,eval_challenges[tid],None,"rejected","malformed_tool_xml")
                        append_record(rec); save_event(tid,event); continue

                    if parsed["kind"]=="tool":
                        calls=parsed.get("tool_calls",[])
                        st["step_retry_count"]=0

                        # Unsupported/malformed structured calls become tool feedback, as in the source loop.
                        malformed=any(tc.get("name")!="python" or "code" not in tc.get("arguments",{}) for tc in calls)
                        if malformed:
                            st["messages"].append(assistant_tool_message(parsed))
                            for tc in calls:
                                st["messages"].append({"role":"tool","content":
                                    f"Tool call rejected: expected python(code=...). Got name={tc.get('name')!r}, keys={list(tc.get('arguments',{}))}."})
                            st["tool_errors"]+=1
                            event["tool_result"]="unsupported_or_invalid_structured_tool_call"
                            save_event(tid,event); continue

                        remaining_tool_calls=MAX_TOOL_CALLS-st["tool_calls"]
                        if len(calls)>remaining_tool_calls:
                            # Keep the conversation structurally valid but do not execute calls beyond the LOCAL budget.
                            st["messages"].append(assistant_tool_message(parsed))
                            denial=("LOCAL P0 tool budget exhausted. No further Python execution is available. "
                                    "Return the self-contained final `solution(grid)` Python code block now.")
                            for _tc in calls:
                                st["messages"].append({"role":"tool","content":denial})
                            st["budget_denials"]+=1
                            event["tool_result"]="tool_budget_denied_finalization_nudge"
                            if st["budget_nudged"]:
                                st["done"]=True
                                rec=complete_record(st,eval_challenges[tid],None,"rejected","tool_budget_exhausted_after_nudge")
                                append_record(rec)
                            else:
                                st["budget_nudged"]=True
                            save_event(tid,event); continue

                        st["messages"].append(assistant_tool_message(parsed))
                        event_calls=[]
                        for tc in calls:
                            code=tc["arguments"]["code"]; st["tool_calls"]+=1
                            rr=workers[tid].execute(code)
                            # worker.restart_count is cumulative within the current source-attempt worker.
                            current_worker_restarts=int(getattr(workers[tid],"restart_count",0))
                            st["worker_restarts"]=st["worker_restart_offset"]+current_worker_restarts
                            if not rr.get("ok"): st["tool_errors"]+=1
                            if rr.get("timeout"): st["tool_timeouts"]+=1
                            raw_tool=str(rr.get("text",""))[:MAX_TOOL_RAW_ARTIFACT_CHARS]
                            model_tool=compact_tool_output(raw_tool)
                            if st["tool_calls"]>=MAX_TOOL_CALLS:
                                model_tool += ("\n\n[LOCAL P0 BUDGET] This was the final permitted Python execution. "
                                               "Your next assistant response must be the final self-contained `solution(grid)` code block.")
                            st["messages"].append({"role":"tool","content":model_tool})
                            event_calls.append({"tool_code":code,"tool_ok":rr.get("ok"),
                                                "tool_raw_output":raw_tool,"tool_output_to_model":model_tool,
                                                "current_worker_restarts":current_worker_restarts})
                        event["tool_calls_executed"]=event_calls
                        save_event(tid,event); continue

                    if parsed["kind"]=="final":
                        st["step_retry_count"]=0
                        code=parsed["final_code"]; st["final_code"]=code
                        verifier=fresh_verify_solution(code,eval_challenges[tid],VERIFIER_TIMEOUT_SEC)
                        if verifier.get("all_train_exact") and verifier.get("test_execution_ok"):
                            cand="verified"
                        elif verifier.get("all_train_exact") and verifier.get("any_test_output_ok"):
                            cand="verified_partial_test"
                        elif verifier.get("all_train_exact"):
                            cand="test_execution_failed"
                        else:
                            cand="train_rejected"
                        st["done"]=True; event["verifier_status"]=verifier.get("verifier_status")
                        rec=complete_record(st,eval_challenges[tid],verifier,cand,"final_program")
                        append_record(rec); save_event(tid,event); continue

                    # No tool and no valid final code: source CodeSolutionTurn rejects and AbstractTurn retries.
                    if start_fresh_attempt(tid,st,"no_tool_call_or_final_solution"):
                        event["source_retry"]=True; save_event(tid,event); continue
                    st["done"]=True
                    rec=complete_record(st,eval_challenges[tid],None,"rejected","no_tool_call_or_final_solution")
                    append_record(rec); save_event(tid,event)

                print(f"batch {batch_no}: active={sum(not states[t]['done'] for t in tids)} | "
                      f"wave={wave_sec:.1f}s | caps={min(max_allowed)}..{max(max_allowed)}")
        finally:
            for tid,w in workers.items():
                try: w.close()
                except Exception: pass

if not USE_MODEL: run_mock_preflight()
else: run_real_piv()
print("Cell 8 complete. Raw checkpoint:",RAW_GENERATIONS_PATH)


In [ ]:
# =====================================================================================
# CELL 9 — CONTRACT-FILTERED COVERAGE + PIV SCORING + VERIFIED CANDIDATE REGISTRY
# =====================================================================================
raw_records=read_existing_records(); compatible_records,checkpoint_audit=audit_checkpoint_records(raw_records)
with open(WORK_ROOT/"checkpoint_audit.json","w") as f: json.dump(checkpoint_audit,f,indent=2)
if not compatible_records: raise RuntimeError("No compatible PIV records")
# Latest terminal record per active task.
latest_map={}
for r in compatible_records:
    if r.get("status") in {"complete","failed"}: latest_map[r["task_id"]]=r
latest=pd.DataFrame([latest_map[t] for t in ACTIVE_TASK_IDS if t in latest_map])
complete_set={t for t,r in latest_map.items() if r.get("status")=="complete"}
failed_set={t for t,r in latest_map.items() if r.get("status")=="failed"}
missing=sorted(ACTIVE_TASK_SET-set(latest_map))
coverage={"complete_tasks":len(complete_set),"failed_tasks":len(failed_set),"expected_tasks":len(ACTIVE_TASK_IDS),
          "missing_tasks":missing,"run_complete":len(missing)==0 and len(failed_set)==0}
run_complete=coverage["run_complete"]
print("Checkpoint audit:",json.dumps(checkpoint_audit,indent=2)); print("Coverage:",json.dumps(coverage,indent=2))

# Score every public-eval test output; missing/rejected program = no PIV candidate.
rows=[]; registry=[]
weight_map=dict(zip(eval_weights["task_id"],eval_weights["weight"]))
quartile_map=dict(zip(eval_weights["task_id"],eval_weights["test_relevance_quartile"]))
for tid in ACTIVE_TASK_IDS:
    r=latest_map.get(tid,{})
    outs=r.get("test_outputs",[]) if r.get("all_train_exact") else []
    for qi,target in enumerate(eval_solutions[tid]):
        pred=outs[qi] if qi<len(outs) and is_rectangular_grid(outs[qi]) else None
        exact=int(pred is not None and grids_equal(pred,target))
        rows.append({"task_id":tid,"test_index":qi,"piv_valid":int(pred is not None),"piv_exact":exact,
                     "train_exact":int(bool(r.get("all_train_exact"))),"candidate_status":r.get("candidate_status"),
                     "weight":weight_map[tid],"quartile":quartile_map[tid]})
        if pred is not None:
            registry.append({"task_id":tid,"test_index":qi,"grid":pred,"canonical_sha256":hashlib.sha256(json.dumps(pred,separators=(",",":"),sort_keys=False).encode()).hexdigest(),
                "source":"qwen3.8_27b_piv_p0","run_id":r.get("run_id"),"record_id":r.get("record_id"),"seed":r.get("seed"),
                "program_sha256":r.get("final_code_sha256"),"train_exact":True,"piv_source_commit":PIV_SOURCE_COMMIT,
                "generation_contract_sha256":r.get("generation_contract_sha256")})
scores=pd.DataFrame(rows); registry_df=pd.DataFrame(registry)
scores.to_csv(WORK_ROOT/"task_output_scores.csv",index=False); registry_df.to_csv(WORK_ROOT/"candidate_registry.csv",index=False)

def weighted_mean(x,w):
    x=np.asarray(x,dtype=float); w=np.asarray(w,dtype=float); return float(np.sum(x*w)/np.sum(w)) if np.sum(w)>0 else float("nan")
raw_acc=float(scores.piv_exact.mean()); weighted_acc=weighted_mean(scores.piv_exact,scores.weight)
valid_rate=float(scores.piv_valid.mean())
train_perfect_rate=float(np.mean([
    bool(latest_map.get(tid,{}).get("all_train_exact")) for tid in ACTIVE_TASK_IDS
]))
mode_summary=pd.DataFrame([{"branch":"piv_p0","output_weighted_exact":raw_acc,"propensity_weighted_diagnostic":weighted_acc,
                            "valid_grid_rate":valid_rate,"train_perfect_program_rate":train_perfect_rate,
                            "tasks":len(ACTIVE_TASK_IDS),"test_outputs":len(scores)}])
mode_summary.to_csv(WORK_ROOT/"mode_summary.csv",index=False)

task_scores=scores.groupby("task_id")["piv_exact"].min().rename("piv_all_queries").reset_index()
task_scores.to_csv(WORK_ROOT/"strict_task_level_scores.csv",index=False)
print(mode_summary.to_string(index=False))
print("STRICT task-normalized exact:",float(task_scores.piv_all_queries.mean()))
print("Verified programs:",int(latest["all_train_exact"].fillna(False).sum()),"/",len(latest))


In [ ]:
# =====================================================================================
# CELL 10 — SOURCE-ALIGNED PROXY / EDA BREAKDOWNS
# =====================================================================================
score_task=scores.groupby("task_id").agg(piv_exact=("piv_exact","mean"),valid=("piv_valid","mean"),train_exact=("train_exact","max")).reset_index()
cat=profile_task_table.merge(score_task,on="task_id",how="left")
CATEGORY_MASKS={
 "ALL":lambda d:np.ones(len(d),dtype=bool),"tier1":lambda d:(d.proxy_tier==1).to_numpy(),"tier2":lambda d:(d.proxy_tier==2).to_numpy(),
 "tier3":lambda d:(d.proxy_tier==3).to_numpy(),"easy_static":lambda d:d.proxy_easy_static.astype(bool).to_numpy(),
 "rare":lambda d:d.proxy_rare.astype(bool).to_numpy(),"multi_query":lambda d:(d.proxy_n_queries>1).to_numpy(),
 "dynamic":lambda d:d.proxy_dynamic.astype(bool).to_numpy(),"high_ratio":lambda d:d.proxy_high_ratio.astype(bool).to_numpy(),
 "extreme_ratio":lambda d:d.proxy_extreme_ratio.astype(bool).to_numpy(),
}
rows=[]
for name,fn in CATEGORY_MASKS.items():
    sub=cat.loc[fn(cat)]
    rows.append({"category":name,"tasks":len(sub),"mean_output_exact":float(sub.piv_exact.mean()) if len(sub) else np.nan,
                 "mean_valid":float(sub.valid.mean()) if len(sub) else np.nan,"train_exact_task_rate":float(sub.train_exact.mean()) if len(sub) else np.nan})
category_scores=pd.DataFrame(rows); category_scores.to_csv(WORK_ROOT/"category_scores.csv",index=False); display(category_scores)


In [ ]:
# =====================================================================================
# CELL 11 — AGENT LOOP / FAILURE TAXONOMY
# =====================================================================================
failure_rows=[]
for _,r in latest.iterrows():
    failure_rows.append({"task_id":r.get("task_id"),"status":r.get("status"),"candidate_status":r.get("candidate_status"),
        "stop_reason":r.get("stop_reason"),"model_turns":r.get("model_turns"),"tool_calls":r.get("tool_calls"),
        "tool_errors":r.get("tool_errors"),"tool_timeouts":r.get("tool_timeouts"),"worker_restarts":r.get("worker_restarts"),
        "generated_tokens":r.get("generated_tokens"),"max_rendered_prompt_tokens":r.get("max_rendered_prompt_tokens"),
        "all_train_exact":r.get("all_train_exact"),"test_execution_ok":r.get("test_execution_ok"),"any_test_output_ok":r.get("any_test_output_ok"),"source_retries":r.get("source_retries"),"step_retries_total":r.get("step_retries_total"),"budget_denials":r.get("budget_denials"),"verifier_status":r.get("verifier_status")})
failure_df=pd.DataFrame(failure_rows); failure_df.to_csv(WORK_ROOT/"agent_diagnostics.csv",index=False)
if len(failure_df):
    print("Candidate status:\n",failure_df.candidate_status.value_counts(dropna=False).to_string())
    print("Stop reason:\n",failure_df.stop_reason.value_counts(dropna=False).to_string())
    print("Mean tool calls:",failure_df.tool_calls.mean(),"| mean generated tokens:",failure_df.generated_tokens.mean())


In [ ]:
# =====================================================================================
# CELL 12 — WRITE SOURCE-STYLE PIV REPLAY PAYLOAD (None when train gate rejects)
# =====================================================================================
pred_piv={}
for tid in ACTIVE_TASK_IDS:
    r=latest_map.get(tid,{})
    outs=r.get("test_outputs",[]) if r.get("all_train_exact") else []
    pred_piv[tid]=[]
    for qi in range(len(eval_challenges[tid]["test"])):
        candidate=outs[qi] if qi<len(outs) and is_rectangular_grid(outs[qi]) else None
        # PiVOTools source leaves attempt_1 empty when final code fails the train gate.
        # We keep None here so no schema filler can ever masquerade as a model candidate in offline analysis.
        pred_piv[tid].append({"attempt_1":candidate,"attempt_2":None})
with open(WORK_ROOT/"predictions_piv_replay.json","w") as f: json.dump(pred_piv,f)
print("Wrote predictions_piv_replay.json")


In [ ]:
# =====================================================================================
# CELL 13 — REPLAY SELF-CHECK: saved source-style payload agrees with candidate exact score
# =====================================================================================
pass1=[]
for tid in ACTIVE_TASK_IDS:
    for qi,target in enumerate(eval_solutions[tid]):
        a=pred_piv[tid][qi]["attempt_1"]
        pass1.append(int(a is not None and grids_equal(a,target)))
replay=float(np.mean(pass1))
assert abs(replay-raw_acc)<1e-12,(replay,raw_acc)
print("Replay self-check PASS:",replay)


In [ ]:
# =====================================================================================
# CELL 14 — PIV METADATA + SOURCE-LEVEL ALIGNMENT REPORT
# =====================================================================================
total_elapsed_sec=time.perf_counter()-notebook_start
source_verification={
 "source_commit":PIV_SOURCE_COMMIT,
 "external_sources":PIV_EXTERNAL_SOURCES,
 "verified_source_files":PIV_SOURCE_FILES,
 "source_faithful":{
   "persistent_repl":True,"puzzle_preloaded":True,"tool_errors_to_model":True,"source_style_retries":True,"self_contained_solution":True,
   "fresh_train_verifier":True,"all_train_exact_gate":True,"separate_test_execution_state":True,"test_outputs_from_execution":True,"single_lineage_p0":True,
 },
 "intentional_local_adaptations":{
   "openai_responses_server":"replaced by direct vLLM + Qwen official tool chat template",
   "docker_ipybox":"replaced by persistent local subprocess + hard kill/restart + Python audit-hook host isolation",
   "source_timeout_120s":"P0 starts at 10s tool / 20s verifier per portability note",
   "source_public_model":"Qwen3.8-27B-FP8, xhigh, preserve_thinking",
   "grid_gate":"strengthened to rectangular integer 0-9 and 1..30 dimensions",
   "budget":"6 executed tool calls, 8 total model turns, 12,288 total generated tokens/task, max 2 source-style retries within the same total budgets",
   "finish_reason_gate":"length/incomplete generations trigger source-style fresh-attempt retry; partial text is never spliced into history",
   "per_turn_ceiling":"4096 tokens for Proxy48; total generated-token cap remains 12288",
   "partial_test_outputs":"train-perfect programs retain successful test outputs even if another test input raises",
   "batch_caps":"per-request max_tokens; no minimum-cap cross-task starvation",
 },
 "not_in_p0":["reviewer","second lineage","PTQ deterministic repair","NVARC seeding","judge","evolution","state compaction ablation"],
}
with open(WORK_ROOT/"source_level_verification.json","w") as f: json.dump(source_verification,f,indent=2)
run_metadata={
 "experiment":EXPERIMENT_ID,"run_profile":RUN_PROFILE,"task_set":TASK_SET,"accuracy_is_real":ACCURACY_IS_REAL,"run_id":RUN_ID,
 "active_task_ids":ACTIVE_TASK_IDS,"proxy48_ordered_sha256":PROXY48_ORDERED_SHA256,"piv_source_commit":PIV_SOURCE_COMMIT,
 "generation_contract_version":GENERATION_CONTRACT_VERSION,"parser_contract_version":PARSER_CONTRACT_VERSION,
 "prompt_contract_version":PROMPT_CONTRACT_VERSION,"seed":SEED,"model_path":str(MODEL_PATH),"model_type":model_cfg.get("model_type"),
 "runtime_stack":RUNTIME_STACK,"model_artifact_fingerprint_sha256":MODEL_ARTIFACT_FINGERPRINT_SHA256,
 "engine_config_sha256":ENGINE_CONFIG_SHA256,"chat_template_sha256":template_sha,"tool_schema_sha256":TOOL_SCHEMA_SHA256,
 "agent_budget":AGENT_BUDGET,"agent_budget_sha256":AGENT_BUDGET_SHA256,"sampling":PIV_SAMPLING,"sampling_config_sha256":SAMPLING_CONFIG_SHA256,
 "model_load_sec":model_load_sec,"warmup_sec":warmup_sec,"total_elapsed_sec":total_elapsed_sec,"max_model_len":MAX_MODEL_LEN,
 "coverage":coverage,"checkpoint_audit":checkpoint_audit,"run_complete":run_complete,
 "output_weighted_exact":raw_acc,"propensity_weighted_diagnostic":weighted_acc,"strict_task_exact":float(task_scores.piv_all_queries.mean()),
 "valid_grid_rate":valid_rate,"train_perfect_program_rate":train_perfect_rate,"candidate_registry_rows":len(registry_df),
 "source_verification":source_verification,
}
with open(WORK_ROOT/"run_metadata.json","w") as f: json.dump(run_metadata,f,indent=2,default=str)

report=[]
report += [f"# {EXPERIMENT_ID} — {RUN_PROFILE}","",f"- Run complete: **{run_complete}**",f"- Accuracy is real: **{ACCURACY_IS_REAL}**",
           f"- Frozen tasks: **{len(ACTIVE_TASK_IDS)}**",f"- PIV source commit: `{PIV_SOURCE_COMMIT}`",
           f"- Output-weighted exact: **{raw_acc:.4f}**",f"- Strict task exact: **{float(task_scores.piv_all_queries.mean()):.4f}**",
           f"- Train-perfect program rate: **{train_perfect_rate:.4f}**",f"- Valid-grid rate: **{valid_rate:.4f}**","",
           "## Causal intervention","Persistent Python REPL + model/tool history + self-contained solution(grid) + fresh all-train exact gate.","",
           "## P0 exclusions","No reviewer, no PTQ repair, no second lineage, no NVARC seeding, no judge, no evolution.","",
           "## Source-level verification","See `source_level_verification.json` and `SOURCE_ALIGNMENT.json` for SOURCE vs LOCAL boundaries."]
(WORK_ROOT/"REPORT_FOR_CHATGPT.md").write_text("\n".join(report),encoding="utf-8")
print("\n".join(report))


In [ ]:
# =====================================================================================
# CELL 15 — MANIFEST + BUNDLE
# =====================================================================================
def sha256_file(path,chunk=1024*1024):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for b in iter(lambda:f.read(chunk),b""): h.update(b)
    return h.hexdigest()
compact_cols=[c for c in ["run_id","record_id","task_id","status","candidate_status","stop_reason","seed","n_train","n_test",
    "model_turns","tool_calls","tool_errors","tool_timeouts","worker_restarts","source_retries","step_retries_total","budget_denials","generated_tokens","max_rendered_prompt_tokens",
    "all_train_exact","test_execution_ok","any_test_output_ok","verifier_status","final_code_sha256","generation_contract_sha256","sampling_config_sha256","agent_budget_sha256",
    "model_artifact_fingerprint_sha256","runtime_stack_sha256","engine_config_sha256","piv_source_commit"] if c in latest.columns]
latest[compact_cols].to_csv(WORK_ROOT/"generation_summary.csv",index=False)
manifest=[]
for p in sorted(WORK_ROOT.rglob('*')):
    if p.is_file() and p.name!="MANIFEST.csv": manifest.append({"file":str(p.relative_to(WORK_ROOT)),"bytes":p.stat().st_size,"sha256":sha256_file(p)})
pd.DataFrame(manifest).to_csv(WORK_ROOT/"MANIFEST.csv",index=False)
ZIP_PATH=Path("/kaggle/working")/BUNDLE_NAME
if ZIP_PATH.exists(): ZIP_PATH.unlink()
with zipfile.ZipFile(ZIP_PATH,"w",compression=zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(WORK_ROOT.rglob('*')):
        if p.is_file(): zf.write(p,arcname=str(p.relative_to(WORK_ROOT)))
print("="*116); print(EXPERIMENT_ID,RUN_PROFILE,"PIPELINE COMPLETE"); print("Bundle:",ZIP_PATH); print("ZIP MB:",round(ZIP_PATH.stat().st_size/1024**2,2))
print("UPLOAD bundle + executed notebook back to ChatGPT.")


## Safe run sequence — PIV-Q27-P0 v1.0

This notebook defaults to:

```python
RUN_PROFILE = "PREFLIGHT"
```

### 1) PREFLIGHT
CPU-safe. It validates the source-aligned tool XML parser, persistent local REPL semantics, puzzle protection, final-code extraction, fresh exact verifier, checkpoint/scoring/bundle paths, and the same pinned tokenizer/runtime-package stack. It does **not** load Qwen3.8-27B and its score is not a model result.

Expected bundle: `PIV_Q27_P0_PREFLIGHT_BUNDLE.zip`.

### 2) SMOKE_GPU
After PREFLIGHT passes, change only:

```python
RUN_PROFILE = "SMOKE_GPU"
```

Use **4×L4, Internet Off**. This runs 8 frozen Proxy tasks with 2 tool calls / 3 model turns to validate real Qwen tool-call rendering, reasoning-history replay, persistent REPL execution, fresh verifier and vLLM batching. This is runtime/interface validation, not the promotion score.

Expected bundle: `PIV_Q27_P0_SMOKE_GPU_BUNDLE.zip`.

### 3) PROXY48
Only after SMOKE_GPU is audited, change only:

```python
RUN_PROFILE = "PROXY48"
```

P0 production budget is frozen at one lineage/task, 6 Python calls, max 7 model turns, max 2,048 generation tokens/turn, max 12,288 generated tokens/task, agent batch 8, 10s tool timeout, 20s fresh verifier timeout, 12h wall-clock with 30m reserve.

Expected bundle: `PIV_Q27_P0_PROXY48_BUNDLE.zip`.

Do not add a reviewer, second lineage, PTQ repair, NVARC hints, judge or evolution to this notebook: that would destroy the causal P0 comparison.
